### Import packeges

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
from scipy.interpolate import griddata
from pathlib import Path
import h5py
import sys
from pathlib import Path

# Where am I running?
try:
    # Normal script
    here = Path(__file__).resolve().parent
except NameError:
    # Notebook / REPL
    here = Path.cwd()

phys_const_path = (here / '..' / 'phys_const').resolve()
sys.path.append(str(phys_const_path))

nsm_plots_path = (here / '..' / 'nsm_plots').resolve()
sys.path.append(str(nsm_plots_path))

nsm_plots_postproc = (here / '..' / 'nsm_instabilities').resolve()
sys.path.append(str(nsm_plots_postproc))

import phys_const as pc
import plot_functions as pf
import functions_angular_crossings as fac

### File path

In [ ]:
##################################################################################################################
# Simulation paths

direct = '/home/erick/gw170817_1.00ye_locsim/CFI/cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_novacmat_cdt_10.0m_att_1e-5'
# direct = '/home/erick/gw170817_1.00ye_locsim/CFI/cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_novacmat_cdt_01.0m_att_1e-4'
# direct = '/home/erick/gw170817_1.00ye_locsim/CFI/cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_novacmat_cdt_00.1m_att_1e-3'
# direct = '/home/erick/gw170817_1.00ye_locsim/CFI/cell_32-48-15_dom_1-1-1_km_ncell_1-1-1_novac_nomat_nonus_cdt_10.0m_att_1e-5'

parfile = '/plt00000_particles'
ncell = (1,1,1) # number of cells in each direction
domain = (1e5, 1e5, 1e5) # cm
num_particles_per_energy_bin = 92
num_energy_bins = 13
direct_interpolated_data = direct

##################################################################################################################

all_par_files = sorted(glob.glob(direct + '/plt*_particles'))
plt_numbers = [f.split('/')[-1].split('plt')[1].split('_particles')[0] for f in all_par_files]
plt_numbers = sorted(plt_numbers, key=lambda x: int(x))
all_par_files = [f'plt{plt_num}_particles' for plt_num in plt_numbers]

finaldir = direct.split('/')[-1]

# Calculate the volume of a single cell and the total volume
cellvolume = (domain[0] / ncell[0]) * (domain[1] / ncell[1]) * (domain[2] / ncell[2]) # cm^3
domainvolume = cellvolume * (ncell[0] * ncell[1] * ncell[2]) # cm^3

### Define cell index to be analyzed

In [ ]:
cell_index_i = 0
cell_index_j = 0
cell_index_k = 0

### Cell to be analized

In [ ]:
# Get the cell indices
cell_file_names = glob.glob(direct + parfile + '/cell_*_*_*')
cell_file_names = [file_name.split('/')[-1] for file_name in cell_file_names]
x_cell_ind = np.array([int(file_name.split('_')[1]) for file_name in cell_file_names])
y_cell_ind = np.array([int(file_name.split('_')[2]) for file_name in cell_file_names])
z_cell_ind = np.array([int((file_name.split('_')[3]).split('.')[0]) for file_name in cell_file_names])
cell_indices = np.array(list(zip(x_cell_ind, y_cell_ind, z_cell_ind)))
print('Number of cells:', len(cell_indices))
print(f'shape of cell_indices: {cell_indices.shape}')

# Get the cell indices for ix fix but iy and iz varying all available cells
x_idx_slice = cell_index_i
mask_yz_slice = cell_indices[:,0] == x_idx_slice # fixing the x index in this value
cell_indices_yz_slice = cell_indices[mask_yz_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_yz_slice[:,1], cell_indices_yz_slice[:,2], color='b')
ax.scatter(cell_index_j, cell_index_k, color='r')
ax.set_xlabel('$i_y$')
ax.set_ylabel('$i_z$')
ax.set_title(f'Cell indices for fixed $i_x=${x_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Get the cell indices for iy fix but ix and iz varying all available cells
y_idx_slice = cell_index_j
mask_xz_slice = cell_indices[:,1] == y_idx_slice # fixing the y index in this value
cell_indices_xz_slice = cell_indices[mask_xz_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_xz_slice[:,0], cell_indices_xz_slice[:,2], color='b')
ax.scatter(cell_index_i, cell_index_k, color='r')
ax.set_xlabel('$i_x$')
ax.set_ylabel('$i_z$')
ax.set_title(f'Cell indices for fixed $i_y=${y_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Get the cell indices for iz fix but ix and iy varying all available cells
z_idx_slice = cell_index_k
mask_xy_slice = cell_indices[:,2] == z_idx_slice # fixing the z index in this value
cell_indices_xy_slice = cell_indices[mask_xy_slice]
fig, ax = plt.subplots(figsize=(12, 8))
ax.scatter(cell_indices_xy_slice[:,0], cell_indices_xy_slice[:,1], color='b')
ax.scatter(cell_index_i, cell_index_j, color='r')
ax.set_xlabel('$i_x$')
ax.set_ylabel('$i_y$')
ax.set_title(f'Cell indices for fixed $i_z=${z_idx_slice}')
ax.legend()
# ax.set_xlim([-5,100])
# ax.set_ylim([-5,70])
plt.show()
plt.close(fig)

# Combine all cell indices from the three slices
# cell_indices_all = np.concatenate((cell_indices_yz_slice, cell_indices_xz_slice, cell_indices_xy_slice), axis=0)
cell_indices_all = np.concatenate((cell_indices_xz_slice, cell_indices_xy_slice), axis=0)
cell_indices_all = np.unique(cell_indices_all, axis=0)

### Getting temperature, electron fraction and density

In [ ]:
rho_ye_T_h5py = h5py.File(direct+'/rho_Ye_T.hdf5', 'r')

Temperature_MeV = np.array(rho_ye_T_h5py['/T_Mev'])[cell_index_i,cell_index_j,cell_index_k]
rho_g_cm3 = np.array(rho_ye_T_h5py['/rho_g|ccm'])[cell_index_i,cell_index_j,cell_index_k]
Ye = np.array(rho_ye_T_h5py['/Ye'])[cell_index_i,cell_index_j,cell_index_k]

print(f'rho_cons = {rho_g_cm3} # g/ccm')
print(f'T_cons = {Temperature_MeV} # MeV')
print(f'Ye_cons = {Ye} # n_electron - n_positron / n_barions')

neutrons_number_density_cm3 = rho_g_cm3 * (1.0 - Ye) / pc.PhysConst.Mp
protons_number_density_cm3 = rho_g_cm3 * Ye / pc.PhysConst.Mp

rho_ye_T_h5py.close()

### Compute ELN number density

In [ ]:
energybinsMeV = np.array([
    1, 3, 5.2382, 8.0097, 11.442, 15.691, 20.953, 27.468,
    35.536, 45.525, 57.895, 73.212, 92.178,
    # 115.66, 144.74, 180.75, 225.33, 280.54
])

energybinstopMeV = np.array([
    2, 4, 6.4765, 9.543, 13.34, 18.042, 23.864, 31.073,
    39.999, 51.052, 64.738, 81.685, 102.67, 
    # 128.65, 160.83, 200.67, 250, 311.08
])

energybinsbottomMeV = np.array([
    0, 2, 4, 6.4765, 9.543, 13.34, 18.042, 23.864, 31.073,
    39.999, 51.052, 64.738, 81.685, 
    # 102.67, 128.65, 160.83, 200.67, 250
])

nu_e_opacities_inv_cm = [1.1279864349828499e-08, 4.7428025628186796e-08, 1.321984832248114e-07, 3.294158114604995e-07, 7.528396092747342e-07, 1.5644452508683583e-06, 2.9487331095815796e-06, 5.1520838790595784e-06, 8.605493196949606e-06, 1.4043114431085709e-05, 2.262451577257334e-05, 3.6135649577238473e-05, 5.731900449884377e-05, 9.036764140607461e-05, 0.00014163060110313624, 0.00022059108455224182, 0.00034117744506685334, 0.0005234543477497763]
nu_x_opacities_inv_cm = [2.174662492044255e-10, 3.0391841308327543e-10, 4.466580398128411e-10, 6.371159236091845e-10, 8.969420647822444e-10, 1.2650799494987814e-09, 1.777602214007199e-09, 2.3794705306483863e-09, 3.149801121595646e-09, 4.208531959563822e-09, 5.307878395583215e-09, 6.9122806344757855e-09, 8.655465405273294e-09, 1.0987165360395685e-08, 1.3820008452372534e-08, 1.7330779319237295e-08, 2.1665700738968832e-08, 2.7031770968769257e-08]
nubar_e_opacities_inv_cm = [1.3566096423813977e-09, 4.4755275717913815e-09, 2.1546754963019424e-08, 5.47531006400814e-08, 1.1123883054511091e-07, 2.042363205567264e-07, 3.5482012940187117e-07, 5.920655243201202e-07, 9.526436748501559e-07, 1.4809548374270669e-06, 2.2289900021493824e-06, 3.2538736229147686e-06, 4.6123319631062746e-06, 6.3540654191625565e-06, 8.519538465802936e-06, 1.1153146580678764e-05, 1.4349950240540414e-05, 1.83608347661424e-05]

nu_e_opacities_inv_cm = np.array(nu_e_opacities_inv_cm)
nu_x_opacities_inv_cm = np.array(nu_x_opacities_inv_cm)
nubar_e_opacities_inv_cm = np.array(nubar_e_opacities_inv_cm)

nu_e_opacities_inv_cm = nu_e_opacities_inv_cm[0:13]
nu_x_opacities_inv_cm = nu_x_opacities_inv_cm[0:13]
nubar_e_opacities_inv_cm = nubar_e_opacities_inv_cm[0:13]

### Define function to get number density as a function of time

In [ ]:
def get_number_density_per_energy_bins(i, j, k, particlefile):

    parcellname = '/cell_' + str(i) + '_' + str(j) + '_' + str(k) + '.h5'

    particles_h5py = h5py.File(direct+particlefile+parcellname, 'r')
    N00_Re    = np.array(particles_h5py['/N00_Re'   ]) # particles
    N00_Rebar = np.array(particles_h5py['/N00_Rebar']) # particles
    N11_Re    = np.array(particles_h5py['/N11_Re'   ]) # particles
    N11_Rebar = np.array(particles_h5py['/N11_Rebar']) # particles
    N22_Re    = np.array(particles_h5py['/N22_Re'   ]) # particles
    N22_Rebar = np.array(particles_h5py['/N22_Rebar']) # particles
    time_s    = np.array(particles_h5py['/time'     ])[0] # seconds
    pupt      = np.array(particles_h5py['/pupt'     ]) # ergs
    pupx     = np.array(particles_h5py['/pupx'     ]) / pupt # unitless
    pupy     = np.array(particles_h5py['/pupy'     ]) / pupt # unitless
    pupz     = np.array(particles_h5py['/pupz'     ]) / pupt # unitless
    energyMeV = pupt / ( 1e6 * pc.CGSUnitsConst.eV ) # MeV     
    particles_h5py.close()

    n00_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n00_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n11_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3 
    n11_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n22_Re_split_energy_bins    = np.zeros(len(energybinsbottomMeV)) # particles / cm^3
    n22_Rebar_split_energy_bins = np.zeros(len(energybinsbottomMeV)) # particles / cm^3

    for t in range(num_energy_bins):
        energymask = (energyMeV >= energybinsbottomMeV[t]) & (energyMeV < energybinstopMeV[t])
        if np.sum(energymask) != num_particles_per_energy_bin:
            print(f"Warning: Energy bin {t} has {np.sum(energymask)} particles, expected {num_particles_per_energy_bin}")
        n00_Re_split_energy_bins   [t] = np.sum(N00_Re[energymask])    / cellvolume # particles / cm^3
        n00_Rebar_split_energy_bins[t] = np.sum(N00_Rebar[energymask]) / cellvolume # particles / cm^3
        n11_Re_split_energy_bins   [t] = np.sum(N11_Re[energymask])    / cellvolume # particles / cm^3
        n11_Rebar_split_energy_bins[t] = np.sum(N11_Rebar[energymask]) / cellvolume # particles / cm^3
        n22_Re_split_energy_bins   [t] = np.sum(N22_Re[energymask])    / cellvolume # particles / cm^3
        n22_Rebar_split_energy_bins[t] = np.sum(N22_Rebar[energymask]) / cellvolume # particles / cm^3

    return n00_Re_split_energy_bins, n00_Rebar_split_energy_bins, n11_Re_split_energy_bins, n11_Rebar_split_energy_bins, n22_Re_split_energy_bins, n22_Rebar_split_energy_bins, time_s

### Plot the time evolution of the neutrino energy spectrum

In [ ]:

dE  = ( energybinstopMeV    - energybinsbottomMeV    ) * ( 1e6 * pc.CGSUnitsConst.eV )    # erg
dE3 = ( energybinstopMeV**3 - energybinsbottomMeV**3 ) * ( 1e6 * pc.CGSUnitsConst.eV )**3 # erg^3
E = ( energybinsMeV ) * ( 1e6 * pc.CGSUnitsConst.eV ) # erg
dOmega = 4.0 * np.pi

phase_space_factor = dOmega * dE * E**2  / pc.PhysConst.hc**3

In [ ]:
all_f00_Re_split_energy_bins    = []
all_f00_Rebar_split_energy_bins = []
all_f11_Re_split_energy_bins    = []
all_f11_Rebar_split_energy_bins = []
all_f22_Re_split_energy_bins    = []
all_f22_Rebar_split_energy_bins = []
tiempo_s = []

for dirpar in all_par_files:
    n00_Re_split_energy_bins, n00_Rebar_split_energy_bins, n11_Re_split_energy_bins, n11_Rebar_split_energy_bins, n22_Re_split_energy_bins, n22_Rebar_split_energy_bins, t_s = get_number_density_per_energy_bins(cell_index_i, cell_index_j, cell_index_k,'/'+dirpar)
    all_f00_Re_split_energy_bins   .append(np.array(n00_Re_split_energy_bins)   / phase_space_factor)
    all_f00_Rebar_split_energy_bins.append(np.array(n00_Rebar_split_energy_bins)/ phase_space_factor)
    all_f11_Re_split_energy_bins   .append(np.array(n11_Re_split_energy_bins)   / phase_space_factor)
    all_f11_Rebar_split_energy_bins.append(np.array(n11_Rebar_split_energy_bins)/ phase_space_factor)
    all_f22_Re_split_energy_bins   .append(np.array(n22_Re_split_energy_bins)   / phase_space_factor)
    all_f22_Rebar_split_energy_bins.append(np.array(n22_Rebar_split_energy_bins)/ phase_space_factor)
    tiempo_s.append(t_s)

print('tiempo_s[-1]=',tiempo_s[-1])
all_f00_Re_split_energy_bins    = np.array(all_f00_Re_split_energy_bins)
all_f00_Rebar_split_energy_bins = np.array(all_f00_Rebar_split_energy_bins)
all_f11_Re_split_energy_bins    = np.array(all_f11_Re_split_energy_bins)
all_f11_Rebar_split_energy_bins = np.array(all_f11_Rebar_split_energy_bins)
all_f22_Re_split_energy_bins    = np.array(all_f22_Re_split_energy_bins)
all_f22_Rebar_split_energy_bins = np.array(all_f22_Rebar_split_energy_bins)
tiempo_s = np.array(tiempo_s)

print(f'energybinsMeV = {energybinsMeV}')
print(f'energybinsMeV.shape = {energybinsMeV.shape}')
print(f'all_f00_Re_split_energy_bins.shape = {all_f00_Re_split_energy_bins.shape}')
print(f'all_f00_Re_split_energy_bins[1].shape = {all_f00_Re_split_energy_bins[1].shape}')
print(f'tiempo_s.shape = {tiempo_s.shape}')

net_absorption_emission_n00_inv_s_inv_cm3 = np.zeros_like(tiempo_s)
net_absorption_emission_n00bar_inv_s_inv_cm3 = np.zeros_like(tiempo_s)

for i in range(len(tiempo_s)):
    net_absorption_emission_n00_inv_s_inv_cm3[i] =  np.sum( (nu_e_opacities_inv_cm * pc.PhysConst.c) * (all_f00_Re_split_energy_bins[i] - all_f00_Re_split_energy_bins[0]) * phase_space_factor )
    net_absorption_emission_n00bar_inv_s_inv_cm3[i] =  np.sum( (nu_e_opacities_inv_cm * pc.PhysConst.c) * (all_f00_Rebar_split_energy_bins[i] - all_f00_Rebar_split_energy_bins[0]) * phase_space_factor )

net_absorption_emission_protons_inv_s_inv_cm3   = +1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3 - net_absorption_emission_n00_inv_s_inv_cm3 )
net_absorption_emission_neutrons_inv_s_inv_cm3  = -1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3 - net_absorption_emission_n00_inv_s_inv_cm3 )
net_absorption_emission_electrons_inv_s_inv_cm3 = -1 * (                                                net_absorption_emission_n00_inv_s_inv_cm3 ) 
net_absorption_emission_positrons_inv_s_inv_cm3 = -1 * ( net_absorption_emission_n00bar_inv_s_inv_cm3                                             ) 

print(f'net_absorption_emission_protons_inv_s_inv_cm3_ = {list(net_absorption_emission_protons_inv_s_inv_cm3)}')
print(f'tiempo_s_ = {list(tiempo_s)}')
print(f'...')

delta_tiempo_s = tiempo_s[1:] - tiempo_s[:-1]
cumulative_net_absorption_emission_protons_inv_s_inv_cm3 = np.cumsum(net_absorption_emission_protons_inv_s_inv_cm3[:-1] * delta_tiempo_s)
cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3 = np.cumsum(net_absorption_emission_neutrons_inv_s_inv_cm3[:-1] * delta_tiempo_s)

Ye_dot_inv_s = ( pc.PhysConst.Mp / rho_g_cm3 ) * ( net_absorption_emission_n00bar_inv_s_inv_cm3 -  net_absorption_emission_n00_inv_s_inv_cm3)
cumulative_Ye_inv_s = np.cumsum(Ye_dot_inv_s[:-1] * delta_tiempo_s)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, net_absorption_emission_n00_inv_s_inv_cm3, label=r'$\dot{n}_{\nu_e}$')
ax.plot(tiempo_s, net_absorption_emission_n00bar_inv_s_inv_cm3, label=r'$\dot{n}_{\bar{\nu}_e}$')
ax.plot(tiempo_s, net_absorption_emission_electrons_inv_s_inv_cm3, label=r'$\dot{n}_{e^-}$')
ax.plot(tiempo_s, net_absorption_emission_positrons_inv_s_inv_cm3, label=r'$\dot{n}_{e^+}$')
ax.plot(tiempo_s, net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$\dot{n}_{n}$')
ax.plot(tiempo_s, net_absorption_emission_protons_inv_s_inv_cm3, label=r'$\dot{n}_{p}$')
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{n}\,[\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.set_xlim(0, 0.005)
ax.set_title('Emission-absorption-driven changes\nwith flavor-conversion rates excluded.\nElectrons, positrons, neutrons,\nand protons are not integrated over time.\nAntineutrinos and neutrinos are integrated over time.\n')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=3, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, net_absorption_emission_protons_inv_s_inv_cm3, label=r'$\dot{n}_{p}$')
ax.plot(tiempo_s, net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$\dot{n}_{n}$')
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{n}\,[\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], cumulative_net_absorption_emission_protons_inv_s_inv_cm3, label=r'$n_{p}$')
ax.plot(tiempo_s[1:], cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$n_{n}$')  
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$n - n_{t_0} \,[\mathrm{cm}^{-3}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], neutrons_number_density_cm3 + cumulative_net_absorption_emission_neutrons_inv_s_inv_cm3, label=r'$n_{n}$')  
ax.plot(tiempo_s[1:], protons_number_density_cm3 + cumulative_net_absorption_emission_protons_inv_s_inv_cm3, label=r'$n_{p}$')
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$n\,[\mathrm{{cm}}^{{-3}}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
# ax.set_xscale('log')
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s, Ye_dot_inv_s)
ax.set_xlabel(r'$t\,[\mathrm{s}]$', fontsize=28)
ax.set_ylabel(r'$\dot{Y}_e\;[\mathrm{s}^{-1}]$', fontsize=28)
leg = ax.legend(framealpha=0.0, fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_Ye_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot for cumulative change in Ye
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], cumulative_Ye_inv_s, color='tab:blue', linewidth=2, marker='', markersize=5)
ax.set_xlabel(r'$t$ [s]', fontsize=28)
ax.set_ylabel(r'$Y_e-Y_e^{t_0}$', fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.95, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.tight_layout()
plt.savefig(f"plots/{finaldir}_delta_Ye_cumulative.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

# Plot for Ye evolution
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s[1:], Ye + cumulative_Ye_inv_s, color='tab:green', linewidth=2, marker='', markersize=5)
ax.set_xlabel(r'$t$ [s]', fontsize=28)
ax.set_ylabel(r'$Y_e$', fontsize=28)
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
leg = ax.legend(framealpha=0.95, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.tight_layout()
plt.savefig(f"plots/{finaldir}_Ye_evolution.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)

In [ ]:
net_absorption_emission_protons_inv_s_inv_cm3_1emin5 = [0.0, -1.072198893452628e+35, -1.52706702434455e+35, -1.743697926635321e+35, -1.8478973558599523e+35, -1.893120886874221e+35, -1.9054432387148474e+35, -1.8987725926918832e+35, -1.8810018748446518e+35, -1.856813264685381e+35, -1.8290736828881673e+35, -1.7995770902079017e+35, -1.769460698867127e+35, -1.739448492224533e+35, -1.7099987604215764e+35, -1.6813960978764248e+35, -1.6538101614729966e+35, -1.6273339583476696e+35, -1.6020092154003888e+35, -1.5778434228945507e+35, -1.5548214126608454e+35, -1.5329132907174134e+35, -1.5120799040107849e+35, -1.4922766190447443e+35, -1.4734559330287582e+35, -1.45556927086376e+35, -1.4385682106603896e+35, -1.4224053062947897e+35, -1.4070346250828585e+35, -1.392412083947497e+35, -1.3784956433450043e+35, -1.3652454012583864e+35, -1.3526236175925625e+35, -1.3405946907551336e+35, -1.3291251020740795e+35, -1.3181833392975735e+35, -1.3077398072272334e+35, -1.2977667312232517e+35, -1.2882380576425127e+35, -1.2791293540461225e+35, -1.2704177111276569e+35, -1.2620816476731092e+35, -1.2541010193906972e+35, -1.246456932121714e+35, -1.2391316596951585e+35, -1.2321085665227738e+35, -1.2253720349121765e+35, -1.218907396992242e+35, -1.2127008710887485e+35, -1.2067395023576524e+35, -1.2010111074588419e+35, -1.1955042230451516e+35, -1.1902080578377225e+35, -1.1851124480620707e+35, -1.1802078160251266e+35, -1.1754851316181325e+35, -1.1709358765466537e+35, -1.1665520110943192e+35, -1.162325943239281e+35, -1.1582504999527743e+35, -1.154318900524302e+35, -1.1505247317611501e+35, -1.1468619249264683e+35, -1.1433247342886692e+35, -1.139907717159678e+35, -1.13660571531628e+35, -1.1334138376977147e+35, -1.1303274442900498e+35, -1.1273421311048943e+35, -1.1244537161753828e+35, -1.1216582264934042e+35, -1.1189518858197505e+35, -1.1163311033018129e+35, -1.1137924628436557e+35, -1.1113327131701026e+35, -1.1089487585389276e+35, -1.1066376500510996e+35, -1.1043965775201928e+35, -1.1022228618584228e+35, -1.1001139479462956e+35, -1.0980673979492266e+35, -1.0960808850534665e+35, -1.0941521875901498e+35, -1.09227918352302e+35, -1.0904598452754722e+35, -1.0886922348739496e+35, -1.0869744993882822e+35, -1.0853048666488634e+35, -1.0836816412250886e+35, -1.082103200647169e+35, -1.0805679918594993e+35, -1.0790745278895915e+35, -1.077621384721876e+35, -1.0762071983668937e+35, -1.0748306621134765e+35, -1.0734905239572374e+35, -1.0721855841969492e+35, -1.0709146931924407e+35, -1.069676749278652e+35, -1.0684706968314783e+35, -1.0672955244804255e+35, -1.0661502634683312e+35, -1.065033986154566e+35, -1.063945804662585e+35, -1.0628848696731564e+35, -1.0618503693656339e+35, -1.0608415285102387e+35, -1.059857607719254e+35, -1.0588979028614877e+35, -1.0579617446516804e+35, -1.0570484984246513e+35, -1.056157564108611e+35, -1.055288376414062e+35, -1.054440405258029e+35, -1.0536131564446648e+35, -1.0528061726315176e+35, -1.0520190346110903e+35, -1.0512513629431346e+35, -1.0505028199804118e+35, -1.049773112335136e+35, -1.0490619938415203e+35, -1.0483692690768718e+35, -1.0476947975167756e+35, -1.0470384984033127e+35, -1.0464003564272889e+35, -1.0457804283289041e+35, -1.045178850547084e+35, -1.0445958480569745e+35, -1.0440317445632695e+35, -1.0434869742343136e+35, -1.0429620951943321e+35, -1.0424578050178351e+35, -1.0419749585068082e+35, -1.0415145880732447e+35, -1.0410779270908593e+35, -1.0406664366376365e+35, -1.0402818361056854e+35, -1.0399261382265359e+35, -1.0396016891361e+35, -1.0393112141919042e+35, -1.0390578703592134e+35, -1.038845306096395e+35, -1.0386777298018792e+35, -1.0385599880386262e+35, -1.0384976549215343e+35, -1.0384971342531314e+35, -1.0385657762152095e+35, -1.038712010680844e+35, -1.0389454995067822e+35, -1.0392773104961028e+35, -1.0397201161058538e+35, -1.0402884204061116e+35, -1.0409988182945847e+35, -1.0418702915366812e+35, -1.0429245468419496e+35, -1.044186401924533e+35, -1.0456842263323616e+35, -1.0474504447767136e+35, -1.0495221117830112e+35, -1.0519415677099829e+35, -1.0547571875874323e+35, -1.058024235810763e+35, -1.0618058415356174e+35, -1.0661741116574027e+35, -1.0712114005770327e+35, -1.0770117585696871e+35, -1.0836825835266562e+35, -1.0913465041654575e+35, -1.1001435265459732e+35, -1.110233479922356e+35, -1.1217988026460298e+35, -1.135047714062343e+35, -1.1502178241260865e+35, -1.167580238861176e+35, -1.1874442267879834e+35, -1.2101625190736928e+35, -1.2361373243785373e+35, -1.2658271481267058e+35, -1.2997545151079312e+35, -1.338514703750126e+35, -1.3827856098042722e+35, -1.4333388661850895e+35, -1.4910523537400918e+35, -1.5569242440331663e+35, -1.6320887187831336e+35, -1.7178335100311684e+35, -1.8156193986087624e+35, -1.9271017937015263e+35, -2.054154490251452e+35, -2.1988956598054187e+35, -2.3637160694781067e+35, -2.551309437131054e+35, -2.7647047116949233e+35, -3.0072999075289443e+35, -3.2828969114166248e+35, -3.595736409856218e+35, -3.950531741928432e+35, -4.352500058925173e+35, -4.80738865783416e+35, -5.32149374780913e+35, -5.901668210676715e+35, -6.555314144312637e+35, -7.290355165293163e+35, -8.115182653445542e+35, -9.038569437031116e+35, -1.0069543973993805e+36, -1.1217218056454779e+36, -1.2490561669509552e+36, -1.3898120119927063e+36, -1.544767116984559e+36, -1.714582388004799e+36, -1.899756629637745e+36, -2.100577592071048e+36, -2.3170714732870995e+36, -2.5489538654514804e+36, -2.7955858673281077e+36, -3.0559395964317023e+36, -3.328577471760769e+36, -3.611649264646893e+36, -3.902909953141246e+36, -4.199759880955844e+36, -4.499306749317379e+36, -4.7984468057528544e+36, -5.0939605585467747e+36, -5.382616768260509e+36, -5.66127760932376e+36, -5.926997886344354e+36, -6.177112004428677e+36, -6.409303859447717e+36, -6.621656666869306e+36, -6.81268169151581e+36, -6.981326613224921e+36, -7.126965677769518e+36, -7.249374742952016e+36, -7.348694826054058e+36, -7.425387844257668e+36, -7.480188005143982e+36, -7.514051853755987e+36, -7.528109414504964e+36, -7.523618262643736e+36, -7.501921782437075e+36, -7.464412357796167e+36, -7.412499817741593e+36, -7.347585130580136e+36, -7.27103910337726e+36, -7.184185686575473e+36, -7.088289393308534e+36, -6.984546304056975e+36, -6.874078125695698e+36, -6.757928797670057e+36, -6.637063177442773e+36, -6.512367385340224e+36, -6.384650440413903e+36, -6.254646870451184e+36, -6.123020028526659e+36, -5.990365894029612e+36, -5.857217177089265e+36, -5.724047581354345e+36, -5.591276111120119e+36, -5.459271335044667e+36, -5.328355540544269e+36, -5.1988087308944084e+36, -5.070872431608907e+36, -4.9447532843347486e+36, -4.820626415743698e+36, -4.6986385761295984e+36, -4.5789110479753615e+36, -4.46154232893136e+36, -4.3466105966963643e+36, -4.234175965429432e+36, -4.124282544730595e+36, -4.016960313066184e+36, -3.912226817913681e+36, -3.8100887149678124e+36, -3.710543158570945e+36, -3.6135790551751627e+36, -3.5191781911625325e+36, -3.427316245786341e+36, -3.3379636993797506e+36, -3.251086646335291e+36, -3.1666475217069195e+36, -3.0846057496410564e+36, -3.0049183212149046e+36, -2.9275403086572104e+36, -2.852425322353643e+36, -2.7795259164994784e+36, -2.7087939487578956e+36, -2.6401808988131413e+36, -2.573638150274471e+36, -2.5091172399869747e+36, -2.4465700784393398e+36, -2.3859491446235194e+36, -2.32720765839623e+36, -2.270299733115006e+36, -2.215180511070948e+36, -2.161806284013582e+36, -2.1101346008597155e+36, -2.060124364495025e+36, -2.011735919413464e+36, -1.9649311317934497e+36, -1.919673463479611e+36, -1.8759280412238252e+36, -1.8336617224363537e+36, -1.792843158608947e+36, -1.7534428574914328e+36, -1.7154332450342433e+36, -1.6787887280480168e+36, -1.6434857584773878e+36, -1.6095029001387538e+36, -1.5768208987290835e+36, -1.545422755874206e+36, -1.5152938079490402e+36, -1.4864218103678394e+36, -1.4587970280072475e+36, -1.4324123323889496e+36, -1.4072633062085745e+36, -1.3833483557520966e+36, -1.3606688316878735e+36, -1.3392291586591298e+36, -1.319036974025253e+36, -1.3001032760071733e+36, -1.2824425813792567e+36, -1.2660730927124586e+36, -1.2510168750069736e+36, -1.2373000413514044e+36, -1.2249529470036517e+36, -1.2140103910000358e+36, -1.2045118240555438e+36, -1.1965015611125713e+36, -1.1900289964187966e+36, -1.1851488184586071e+36, -1.1819212214172158e+36, -1.1804121091138729e+36, -1.1806932864906879e+36, -1.1828426327800964e+36, -1.1869442493907261e+36, -1.1930885743440646e+36, -1.201372453765648e+36, -1.2118991594880956e+36, -1.2247783402732883e+36, -1.2401258925295856e+36, -1.2580637347184312e+36, -1.2787194679631788e+36, -1.3022259037529807e+36, -1.3287204381610196e+36, -1.3583442507759965e+36, -1.391241305708199e+36, -1.4275571317350847e+36, -1.4674373590791793e+36, -1.5110259916712293e+36, -1.5584633962723498e+36, -1.6098839937496386e+36, -1.6654136433598766e+36, -1.7251667183166767e+36, -1.7892428803767401e+36, -1.8577235727914944e+36, -1.9306682647382132e+36, -2.0081104961347635e+36, -2.090053789247029e+36, -2.1764675121921775e+36, -2.2672827985679915e+36, -2.3623886459913737e+36, -2.461628333064116e+36, -2.564796307762337e+36, -2.671635708901107e+36, -2.7818366845739555e+36, -2.895035665846523e+36, -3.0108157393191413e+36, -3.128708237750078e+36, -3.248195633666829e+36, -3.36871577746732e+36, -3.4896674704481317e+36, -3.6104173068589275e+36, -3.7303076605776694e+36, -3.8486656349914946e+36, -3.96481274307793e+36, -4.0780750423537594e+36, -4.187793419680488e+36, -4.2933337064509467e+36, -4.394096306892197e+36, -4.489525041263329e+36, -4.579114940424449e+36, -4.662418776176497e+36, -4.73905216948698e+36, -4.80869718213105e+36, -4.8711043620405837e+36, -4.9260932745874876e+36, -4.9735516074869667e+36, -5.0134329831800094e+36, -5.0457536476211794e+36, -5.070588227583539e+36, -5.088064760088247e+36, -5.098359198372438e+36, -5.10168959051145e+36, -5.0983101113434753e+36, -5.0885051077600573e+36, -5.0725832937085496e+36, -5.050872206174971e+36, -5.023713008458697e+36, -4.991455703358181e+36, -4.954454797278306e+36, -4.913065437260358e+36, -4.8676400267853754e+36, -4.818525312966576e+36, -4.7660599273275864e+36, -4.710572354550398e+36, -4.652379298095543e+36, -4.591784408130171e+36, -4.529077335422112e+36, -4.4645330744532275e+36, -4.398411559673751e+36, -4.3309574802972234e+36, -4.2624002810887156e+36, -4.1929543190335243e+36, -4.122819148426653e+36, -4.052179909669246e+36, -3.981207799795824e+36, -3.910060605413021e+36, -3.83888328125358e+36, -3.767808559902342e+36, -3.696957580414218e+36, -3.626440525506085e+36, -3.5563572587634134e+36, -3.486797954863708e+36, -3.417843717189899e+36, -3.349567178401048e+36, -3.282033080557505e+36, -3.2152988322790624e+36, -3.149415041160693e+36, -3.0844260202979376e+36, -3.020370268294743e+36, -2.957280922555053e+36, -2.895186186007338e+36, -2.8341097276892984e+36, -2.7740710578384737e+36, -2.715085878302284e+36, -2.657166409205967e+36, -2.600321692905371e+36, -2.5445578763109256e+36, -2.4898784727027836e+36, -2.4362846041717962e+36, -2.3837752258179343e+36, -2.3323473328232137e+36, -2.2819961514900764e+36, -2.2327153153032305e+36, -2.184497027033705e+36, -2.1373322078601633e+36, -2.0912106344367074e+36, -2.0461210647878842e+36, -2.002051353863532e+36, -1.9589885595363374e+36, -1.9169190397774112e+36, -1.8758285416974573e+36, -1.835702283095093e+36, -1.7965250271097053e+36, -1.758281150533373e+36, -1.7209547062961895e+36, -1.6845294806002057e+36, -1.648989045141435e+36, -1.614316804824432e+36, -1.5804960413420876e+36, -1.547509952962959e+36, -1.5153416908404318e+36, -1.4839743921315587e+36, -1.453391210189618e+36, -1.4235753420711843e+36, -1.3945100535785332e+36, -1.366178702038459e+36, -1.3385647570011167e+36, -1.3116518190264583e+36, -1.2854236367108735e+36, -1.2598641220930449e+36, -1.2349573645659871e+36, -1.2106876434103285e+36, -1.1870394390544709e+36, -1.1639974431571315e+36, -1.1415465675999482e+36, -1.1196719524697875e+36, -1.0983589731036439e+36, -1.0775932462627937e+36, -1.057360635496991e+36, -1.0376472557548108e+36, -1.0184394772911946e+36, -9.997239289195328e+35, -9.814875006516159e+35, -9.637173457657492e+35, -9.464008823400082e+35, -9.295257942851537e+35, -9.130800319093192e+35, -8.970518120443146e+35, -8.814296177614798e+35, -8.66202197703249e+35, -8.513585650553266e+35, -8.368879961823027e+35, -8.227800289490336e+35, -8.090244607484767e+35, -7.956113462556665e+35, -7.825309949269213e+35, -7.697739682620675e+35, -7.57331076846979e+35, -7.451933771929952e+35, -7.333521683889717e+35, -7.217989885812962e+35, -7.105256112964132e+35, -6.9952404162018e+35, -6.887865122475376e+35, -6.783054794156619e+35, -6.680736187334516e+35, -6.580838209193867e+35, -6.483291874597164e+35, -6.388030261984382e+35, -6.294988468697927e+35, -6.204103565842347e+35, -6.115314552777368e+35, -6.028562311343588e+35, -5.9437895599138186e+35, -5.860940807361369e+35, -5.779962307028155e+35, -5.700802010776089e+35, -5.623409523200362e+35, -5.5477360560747565e+35, -5.473734383102109e+35, -5.401358795033136e+35, -5.330565055214866e+35, -5.2613103556283945e+35, -5.193553273465777e+35, -5.127253728298923e+35, -5.062372939882775e+35, -4.998873386638439e+35, -4.936718764848517e+35, -4.8758739486047154e+35, -4.816304950533826e+35, -4.7579788833319085e+35, -4.700863922128174e+35, -4.644929267700143e+35, -4.5901451105570405e+35, -4.5364825959052e+35, -4.4839137895069444e+35, -4.432411644440544e+35, -4.381949968768688e+35, -4.332503394117266e+35, -4.2840473451657915e+35, -4.2365580100485e+35, -4.190012311663554e+35, -4.144387879883164e+35, -4.0996630246611646e+35, -4.055816710025652e+35, -4.012828528950084e+35, -3.970678679090136e+35, -3.92934793937404e+35, -3.888817647433768e+35, -3.849069677860882e+35, -3.8100864212747825e+35, -3.7718507641845626e+35, -3.734346069629387e+35, -3.697556158580839e+35, -3.661465292088157e+35, -3.626058154150206e+35, -3.591319835294758e+35, -3.557235816847974e+35, -3.523791955874446e+35, -3.4909744707698265e+35, -3.45876992748783e+35, -3.4271652263825106e+35, -3.39614758964594e+35, -3.365704549326515e+35, -3.335823935905479e+35, -3.3064938674150073e+35, -3.2777027390807507e+35, -3.2494392134692016e+35, -3.2216922111232466e+35, -3.1944509016679763e+35, -3.167704695369632e+35, -3.141443235130426e+35, -3.115656388902808e+35, -3.09033424250558e+35, -3.0654670928279898e+35, -3.0410454414020965e+35, -3.017059988331296e+35, -2.993501626557238e+35, -2.9703614364511806e+35, -2.9476306807139068e+35, -2.9253007995701646e+35, -2.9033634062441854e+35, -2.8818102827006902e+35, -2.8606333756390695e+35, -2.839824792726725e+35, -2.819376799059081e+35, -2.799281813833376e+35, -2.7795324072243747e+35, -2.7601212974493256e+35, -2.7410413480111592e+35, -2.722285565108806e+35, -2.703847095204211e+35, -2.6857192227337207e+35, -2.6678953679561573e+35, -2.6503690849273673e+35, -2.633134059589833e+35, -2.6161841079717193e+35, -2.599513174484569e+35, -2.583115330311812e+35, -2.5669847718818013e+35, -2.5511158194161754e+35, -2.535502915547238e+35, -2.5201406239991865e+35, -2.505023628324312e+35, -2.4901467306909508e+35, -2.4755048507161052e+35, -2.4610930243380284e+35, -2.4469064027244565e+35, -2.4329402512121895e+35, -2.4191899482725284e+35, -2.4056509845001187e+35, -2.3923189616225496e+35, -2.3791895915245915e+35, -2.3662586952876244e+35, -2.353522202238866e+35, -2.3409761490096584e+35, -2.3286166786003678e+35, -2.316440039448728e+35, -2.304442584502289e+35, -2.292620770290267e+35, -2.280971155996787e+35, -2.269490402530977e+35, -2.2581752715952968e+35, -2.2470226247495857e+35, -2.236029422470278e+35, -2.2251927232041888e+35, -2.214509682416147e+35, -2.2039775516281407e+35, -2.193593677452193e+35, -2.1833555006126105e+35, -2.173260554960677e+35, -2.1633064664767334e+35, -2.1534909522639837e+35, -2.1438118195277932e+35, -2.134266964543598e+35, -2.1248543716104994e+35, -2.11557211199096e+35, -2.1064183428329326e+35, -2.0973913060764014e+35, -2.0884893273408465e+35, -2.079710814792771e+35, -2.0710542579925305e+35, -2.0625182267174376e+35, -2.0541013697610256e+35, -2.0458024137058647e+35, -2.0376201616676138e+35, -2.029553492009562e+35, -2.0216013570241433e+35, -2.0137627815815322e+35, -2.0060368617394913e+35, -1.9984227633170645e+35, -1.9909197204248815e+35, -1.9835270339537815e+35, -1.9762440700176483e+35, -1.9690702583478917e+35, -1.962005090638922e+35, -1.9550481188407063e+35, -1.948198953398178e+35, -1.9414572614341495e+35, -1.9348227648748172e+35, -1.92829523851682e+35, -1.9218745080321848e+35, -1.915560447913754e+35, -1.9093529793566337e+35, -1.9032520680787866e+35, -1.897257722076714e+35, -1.89136998931981e+35, -1.885588955383073e+35, -1.87991474101791e+35, -1.8743474996646105e+35, -1.8688874149072297e+35, -1.8635346978747547e+35, -1.8582895845895518e+35, -1.853152333269639e+35, -1.848123221585907e+35, -1.843202543882552e+35, -1.8383906083626957e+35, -1.833687734248397e+35, -1.829094248918644e+35, -1.8246104850348737e+35, -1.820236777660905e+35, -1.815973461384795e+35, -1.8118208674521535e+35, -1.8077793209199672e+35, -1.803849137840433e+35, -1.8000306224845364e+35, -1.7963240646156897e+35, -1.792729736824244e+35, -1.7892478919330652e+35, -1.7858787604850826e+35, -1.7826225483233067e+35, -1.7794794342748456e+35, -1.7764495679486732e+35, -1.773533067657056e+35, -1.770730018471695e+35, -1.768040470423858e+35, -1.7654644368566076e+35, -1.763001892938134e+35, -1.760652774344672e+35, -1.7584169761186e+35, -1.756294351708961e+35, -1.7542847121976253e+35, -1.7523878257188678e+35, -1.7506034170710153e+35, -1.7489311675251716e+35, -1.747370714831073e+35, -1.7459216534192508e+35, -1.7445835347978928e+35, -1.743355868141895e+35, -1.7422381210698933e+35, -1.7412297206037964e+35, -1.740330054304617e+35, -1.73953847157592e+35, -1.7388542851277285e+35, -1.7382767725889368e+35, -1.7378051782593506e+35, -1.7374387149873644e+35, -1.7371765661623798e+35, -1.7370178878078873e+35, -1.7369618107599413e+35, -1.737007442917736e+35, -1.7371538715506356e+35, -1.7374001656460763e+35, -1.7377453782824668e+35, -1.7381885490107653e+35, -1.7387287062307452e+35, -1.7393648695436753e+35, -1.7400960520678157e+35, -1.7409212627005435e+35, -1.7418395083131477e+35, -1.7428497958631113e+35, -1.7439511344123583e+35, -1.7451425370347747e+35, -1.7464230226068124e+35, -1.747791617463873e+35, -1.7492473569172343e+35, -1.7507892866192643e+35, -1.7524164637699273e+35, -1.7541279581582693e+35, -1.755922853030628e+35, -1.7578002457833052e+35, -1.759759248474083e+35, -1.7617989881507513e+35, -1.7639186069949914e+35, -1.7661172622810904e+35, -1.768394126150228e+35, -1.7707483852022032e+35, -1.7731792399063425e+35, -1.7756859038374838e+35, -1.7782676027383874e+35, -1.7809235734179467e+35, -1.7836530624879723e+35, -1.786455324948652e+35, -1.789329622628413e+35, -1.792275222488134e+35, -1.7952913947996065e+35, -1.79837741120498e+35, -1.8015325426722436e+35, -1.8047560573542276e+35, -1.8080472183629777e+35, -1.8114052814737076e+35, -1.8148294927671038e+35, -1.818319086225578e+35, -1.8218732812933513e+35, -1.8254912804165376e+35, -1.8291722665736515e+35, -1.8329154008119702e+35, -1.8367198198015847e+35, -1.840584633422443e+35, -1.844508922396809e+35, -1.848491735982022e+35, -1.8525320897355615e+35, -1.8566289633679516e+35, -1.8607812986966752e+35, -1.8649879977128085e+35, -1.869247920777102e+35, -1.873559884955182e+35, -1.8779226625072633e+35, -1.8823349795438296e+35, -1.8867955148600193e+35, -1.891302898960031e+35, -1.895855713285174e+35, -1.9004524896525733e+35, -1.9050917099189707e+35, -1.9097718058772283e+35, -1.9144911593952996e+35, -1.919248102804985e+35, -1.9240409195499697e+35, -1.9288678450971497e+35, -1.933727068118147e+35, -1.93861673194551e+35, -1.943534936305293e+35, -1.9484797393299087e+35, -1.953449159849151e+35, -1.958441179961333e+35, -1.9634537478789018e+35, -1.9684847810471613e+35, -1.973532169527486e+35, -1.978593779640822e+35, -1.9836674578579906e+35, -1.9887510349290093e+35, -1.9938423302366957e+35, -1.998939156359055e+35, -2.004039323824953e+35, -2.009140646043021e+35, -2.014240944383602e+35, -2.0193380533904483e+35, -2.0244298260998318e+35, -2.0295141394380128e+35, -2.0345888996727596e+35, -2.0396520478868304e+35, -2.0447015654454615e+35, -2.0497354794257324e+35, -2.054751867973525e+35, -2.0597488655581756e+35, -2.064724668087696e+35, -2.0696775378532667e+35, -2.074605808266682e+35, -2.079507888357321e+35, -2.084382266997478e+35, -2.0892275168192874e+35, -2.094042297795339e+35, -2.0988253604503266e+35, -2.1035755486770096e+35, -2.10829180212829e+35, -2.112973158162162e+35, -2.1176187533170932e+35, -2.1222278242996814e+35, -2.1267997084674596e+35, -2.13133384379533e+35, -2.135829768315653e+35, -2.140287119027458e+35, -2.1447056302721075e+35, -2.1490851315788407e+35, -2.1534255449857464e+35, -2.1577268818480382e+35, -2.1619892391463188e+35, -2.1662127953161884e+35, -2.170397805620161e+35, -2.174544597089549e+35, -2.1786535630680438e+35, -2.1827251573889945e+35, -2.186759888227339e+35, -2.1907583116646333e+35, -2.194721025013639e+35, -2.198648659946298e+35, -2.202541875477481e+35, -2.206401350852038e+35, -2.210227778390404e+35, -2.214021856343933e+35, -2.217784281815904e+35, -2.2215157438013333e+35, -2.225216916401337e+35, -2.228888452264226e+35, -2.2325309763066952e+35, -2.236145079766549e+35, -2.2397313146360394e+35, -2.243290188521642e+35, -2.246822159976337e+35, -2.250327634344668e+35, -2.2538069601585634e+35, -2.2572604261173766e+35, -2.2606882586844556e+35, -2.2640906203231553e+35, -2.2674676083971413e+35, -2.270819254748501e+35, -2.2741455259704906e+35, -2.277446324376045e+35, -2.2807214896723527e+35, -2.2839708013311817e+35, -2.287193981654253e+35, -2.2903906995171424e+35, -2.2935605747772792e+35, -2.296703183322888e+35, -2.2998180627379556e+35, -2.3029047185515898e+35, -2.3059626310378424e+35, -2.3089912625264935e+35, -2.311990065183426e+35, -2.3149584892141354e+35, -2.3178959914402592e+35, -2.3208020441987422e+35, -2.3236761445086544e+35, -2.3265178234471942e+35, -2.329326655677882e+35, -2.332102269068962e+35, -2.334844354341069e+35, -2.337552674679443e+35, -2.340227075250096e+35, -2.342867492551976e+35, -2.345473963545299e+35, -2.3480466344882044e+35, -2.3505857694222754e+35, -2.353091758242108e+35, -2.355565124289878e+35, -2.358006531414599e+35, -2.360416790438184e+35, -2.36279686497458e+35, -2.3651478765467873e+35, -2.3674711089542856e+35, -2.3697680118401028e+35, -2.3720402034165104e+35, -2.37428947230656e+35, -2.3765177784654492e+35, -2.37872725314788e+35, -2.380920197892093e+35, -2.383099082496559e+35, -2.38526654196779e+35, -2.3874253724234863e+35, -2.389578525941053e+35, -2.391729104342778e+35, -2.393880351920169e+35, -2.3960356470965716e+35, -2.3981984930423526e+35, -2.4003725072525803e+35, -2.402561410110249e+35, -2.404769012459141e+35, -2.4069992022170035e+35, -2.409255930065081e+35, -2.4115431942582855e+35, -2.4138650245980823e+35, -2.416225465627813e+35, -2.418628559101351e+35, -2.421078325793869e+35, -2.423578746720144e+35, -2.426133743834938e+35, -2.428747160292818e+35, -2.431422740351721e+35, -2.4341641090050686e+35, -2.436974751435855e+35, -2.439857992386484e+35, -2.4428169755419235e+35, -2.445854643029949e+35, -2.4489737151404563e+35, -2.452176670369532e+35, -2.455465725898564e+35, -2.4588428186154607e+35, -2.4623095867877448e+35, -2.4658673524979045e+35, -2.4695171049501798e+35, -2.4732594847548012e+35, -2.4770947692979858e+35, -2.4810228592999402e+35, -2.4850432666603473e+35, -2.4891551036874785e+35, -2.493357073803804e+35, -2.497647463810793e+35, -2.5020241377946992e+35, -2.5064845327454137e+35, -2.5110256559522435e+35, -2.515644084233126e+35, -2.5203359650468537e+35, -2.5250970195223776e+35, -2.5299225474364257e+35, -2.534807434153857e+35, -2.539746159537704e+35, -2.544732808822528e+35, -2.549761085433911e+35, -2.554824325723832e+35, -2.5599155155806875e+35, -2.565027308859565e+35, -2.5701520475660982e+35, -2.5752817837171875e+35, -2.5804083027868237e+35, -2.585523148638026e+35, -2.590617649827197e+35, -2.5956829471606257e+35, -2.6007100223708415e+35, -2.605689727772739e+35, -2.6106128167525478e+35, -2.615469974935572e+35, -2.620251851871242e+35, -2.6249490930735892e+35, -2.6295523722475648e+35, -2.6340524235331654e+35, -2.638440073596127e+35, -2.6427062733977784e+35, -2.646842129473173e+35, -2.650838934556517e+35, -2.6546881973933256e+35, -2.658381671583498e+35, -2.6619113833117883e+35, -2.6652696578234242e+35, -2.6684491445178596e+35, -2.6714428405393966e+35, -2.6742441127568258e+35, -2.676846718035237e+35, -2.6792448217146552e+35, -2.6814330142247296e+35, -2.6834063257763058e+35, -2.685160239084839e+35, -2.686690700095458e+35, -2.687994126690436e+35, -2.689067415375818e+35, -2.6899079459570053e+35, -2.6905135842251422e+35, -2.6908826826879426e+35, -2.6910140793960678e+35, -2.690907094917888e+35, -2.6905615275377896e+35, -2.6899776467525312e+35, -2.6891561851583687e+35, -2.688098328823123e+35, -2.6868057062490496e+35, -2.6852803760380695e+35, -2.6835248133741665e+35, -2.681541895446883e+35, -2.679334885937872e+35, -2.6769074187021237e+35, -2.67426348076777e+35, -2.6714073947906106e+35, -2.668343801088123e+35, -2.6650776393848864e+35, -2.661614130394292e+35, -2.6579587573606842e+35, -2.6541172476832263e+35, -2.65009555473413e+35, -2.6458998399834085e+35, -2.6415364555334895e+35, -2.6370119271603073e+35, -2.632332937950173e+35, -2.6275063126151745e+35, -2.622539002559136e+35, -2.6174380717577107e+35, -2.6122106835074108e+35, -2.606864088088692e+35, -2.6014056113748442e+35, -2.5958426444148386e+35, -2.590182633998154e+35, -2.584433074209296e+35, -2.5786014989613586e+35, -2.5726954754908513e+35, -2.5667225987865636e+35, -2.5606904869121803e+35, -2.5546067771771487e+35, -2.5484791230963696e+35, -2.542315192076002e+35, -2.536122663748075e+35, -2.5299092288795373e+35, -2.5236825887633013e+35, -2.5174504550019465e+35, -2.511220549588465e+35, -2.5050006051826923e+35, -2.498798365480936e+35, -2.4926215855766247e+35, -2.4864780322070367e+35, -2.480375483784068e+35, -2.4743217301068073e+35, -2.4683245716596674e+35, -2.4623918184012768e+35, -2.4565312879563746e+35, -2.4507508031273326e+35, -2.4450581886512585e+35, -2.4394612671343144e+35, -2.4339678541040977e+35, -2.4285857521315687e+35, -2.423322743981002e+35, -2.418186584758732e+35, -2.4131849930428776e+35, -2.4083256409816195e+35, -2.403616143366179e+35, -2.3990640456881363e+35, -2.394676811205202e+35, -2.3904618070489726e+35, -2.3864262894162978e+35, -2.382577387898295e+35, -2.3789220890065533e+35, -2.375467218965112e+35, -2.3722194258452965e+35, -2.3691851611245277e+35, -2.366370660757629e+35, -2.3637819258520336e+35, -2.3614247030438e+35, -2.3593044646721306e+35, -2.35742638885219e+35, -2.3557953395482146e+35, -2.3544158467460914e+35, -2.353292086826917e+35, -2.3524278632368807e+35, -2.351826587550506e+35, -2.3514912610186896e+35, -2.3514244566884644e+35, -2.35162830217939e+35, -2.352104463194705e+35, -2.352854127840775e+35, -2.353877991824233e+35, -2.3551762445881858e+35, -2.3567485564475056e+35, -2.3585940667724537e+35, -2.360711373271062e+35, -2.3630985224100923e+35, -2.3657530010139393e+35, -2.368671729073878e+35, -2.3718510537991638e+35, -2.3752867449347262e+35, -2.378973991371942e+35, -2.3829073990714664e+35, -2.3870809903215822e+35, -2.3914882043482712e+35, -2.396121899297325e+35, -2.4009743556061265e+35, -2.4060372807829087e+35, -2.41130181561317e+35, -2.4167585418121824e+35, -2.422397491143035e+35, -2.4282081560237308e+35, -2.434179501642098e+35, -2.440299979602265e+35, -2.446557543127242e+35, -2.4529396638370587e+35, -2.459433350127274e+35, -2.466025167168087e+35, -2.47270125854373e+35, -2.479447369548792e+35, -2.486248872155526e+35, -2.493090791661842e+35, -2.499957835023653e+35, -2.50683442087118e+35, -2.513704711198332e+35, -2.520552644712173e+35, -2.5273619718130738e+35, -2.5341162911722406e+35, -2.5407990878606294e+35, -2.5473937729696994e+35, -2.553883724656222e+35, -2.560252330526101e+35, -2.5664830312632275e+35, -2.5725593653906008e+35, -2.5784650150418553e+35, -2.584183852604211e+35, -2.589699988080619e+35, -2.5949978170056607e+35, -2.6000620687369956e+35, -2.604877854929615e+35, -2.6094307179921512e+35, -2.613706679309498e+35, -2.6176922870117246e+35, -2.6213746630568106e+35, -2.624741549392445e+35, -2.6277813529558025e+35, -2.630483189268126e+35, -2.6328369243822024e+35, -2.6348332149400247e+35, -2.6364635461063692e+35, -2.637720267147033e+35, -2.6385966244309343e+35, -2.639086791647554e+35, -2.6391858970439686e+35, -2.6388900475016366e+35, -2.6381963492914316e+35, -2.63710292536483e+35, -2.6356089290622533e+35, -2.633714554139707e+35, -2.6314210410428097e+35, -2.628730679378144e+35, -2.625646806561994e+35, -2.6221738026484487e+35, -2.6183170813681364e+35, -2.6140830774303034e+35, -2.609479230170009e+35, -2.6045139636418988e+35, -2.5991966632856497e+35, -2.5935376493097986e+35, -2.587548146955872e+35, -2.581240253824116e+35, -2.574626904454574e+35, -2.5677218323694475e+35, -2.560539529790507e+35, -2.5530952052534432e+35, -2.545404739344057e+35, -2.537484638782058e+35, -2.529351989079885e+35, -2.521024405997802e+35, -2.5125199860137473e+35, -2.503857256019034e+35, -2.4950551224421783e+35, -2.4861328199962576e+35, -2.4771098602298858e+35, -2.468005980056657e+35, -2.4588410904220975e+35, -2.4496352252572256e+35, -2.4404084908568437e+35, -2.4311810158096478e+35, -2.4219729015939605e+35, -2.4128041739495073e+35, -2.4036947351184974e+35, -2.3946643170490467e+35, -2.38573243564247e+35, -2.3769183461222732e+35, -2.368240999596728e+35, -2.3597190008855813e+35, -2.3513705676753973e+35, -2.3432134910684487e+35, -2.3352650975893445e+35, -2.327542212708888e+35, -2.3200611259497068e+35, -2.3128375576328614e+35, -2.3058866273271795e+35, -2.2992228240615448e+35, -2.252276628482585e+35, -2.241243226108175e+35, -2.2679868445247674e+35, -2.327866013200412e+35, -2.4110460531695124e+35, -2.504256408918167e+35, -2.593002818881683e+35, -2.6646288241640893e+35, -2.711792382484048e+35, -2.734688480009727e+35, -2.740594432172909e+35, -2.741111368282575e+35, -2.7488156600832847e+35, -2.7747466838045203e+35, -2.827046217706039e+35, -2.910258137384658e+35, -3.024633168543769e+35, -3.1651761447891467e+35, -3.3209463682134714e+35, -3.4757516986036475e+35, -3.611030796009746e+35, -3.710213504993452e+35, -3.762458003057237e+35, -3.763982059932959e+35, -3.717069501375785e+35, -3.628355769095607e+35, -3.50775054924276e+35, -3.367842795691816e+35, -3.222667860602804e+35, -3.0853983610245664e+35, -2.9658740931568798e+35, -2.8691920203771805e+35, -2.795709590687864e+35, -2.7420141731166212e+35, -2.7023405696812163e+35, -2.670168985301354e+35, -2.6397752526871638e+35, -2.607340132161048e+35, -2.5712368688315034e+35, -2.5315124626727963e+35, -2.4890671166776453e+35, -2.4451241443238808e+35, -2.4011869447947144e+35, -2.3592196216396884e+35, -2.3216399134918253e+35, -2.2908845116802904e+35, -2.268625117727352e+35, -2.2550098772804265e+35, -2.248391545212541e+35, -2.2457589089596184e+35, -2.2436646089408588e+35, -2.2391826136711264e+35, -2.2305025459024935e+35, -2.217063600432434e+35, -2.1994292378095057e+35, -2.1792027610563863e+35, -2.1590875396954097e+35, -2.1428480634747204e+35, -2.1347641259058708e+35, -2.1384222978017175e+35, -2.1552386749108336e+35, -2.1835159483058712e+35, -2.2186697597304455e+35, -2.2545433108867316e+35, -2.285069154326714e+35, -2.3055226387209844e+35, -2.3131552098368145e+35, -2.3074761924978874e+35, -2.2904534781914304e+35, -2.2665936168414936e+35, -2.2426470404310392e+35, -2.2267392130128766e+35, -2.2269522622633602e+35, -2.2496118121583703e+35, -2.2976734725043873e+35, -2.369643200719977e+35, -2.4594176413314363e+35, -2.557252557286672e+35, -2.6517535613674693e+35, -2.7324277997201834e+35, -2.7920144356293732e+35, -2.827670627830004e+35, -2.840479772789402e+35, -2.8337059528930898e+35, -2.8109763818287648e+35, -2.7753861196299106e+35, -2.7296750868112914e+35, -2.6769424119263297e+35, -2.6212147859775062e+35, -2.5674386618132028e+35, -2.5208285152295218e+35, -2.485829800407096e+35, -2.465211624160916e+35, -2.4597455005696287e+35, -2.4683867542982197e+35, -2.4884196797259576e+35, -2.5153616117917727e+35, -2.543098418063767e+35, -2.5647400479821165e+35, -2.5740945708747646e+35, -2.5672375314295533e+35, -2.5436310383444912e+35, -2.5063412737196946e+35, -2.461053815186562e+35, -2.4140849298332576e+35, -2.370386849982259e+35, -2.332816619129487e+35, -2.302993400261824e+35, -2.2826410627915433e+35, -2.273996423864703e+35, -2.2790311674418648e+35, -2.2984582194002193e+35, -2.3314186060699077e+35, -2.375798325730954e+35, -2.4287697638513334e+35, -2.48769620195478e+35, -2.551708194483608e+35, -2.6232384028079266e+35, -2.707652193156728e+35, -2.8096794740174748e+35, -2.9278853030404273e+35, -3.0507401253272152e+35, -3.1577854368762497e+35, -3.2269917821218735e+35, -3.2457931203545557e+35, -3.2202126540812668e+35, -3.1766414197708212e+35, -3.1547907627477078e+35, -3.194888815813397e+35, -3.3232509507473015e+35, -3.538896352258355e+35, -3.8054526252857784e+35, -4.056196792692416e+35, -4.216580199206634e+35, -4.2360691364769235e+35, -4.111572855315333e+35, -3.888722891670088e+35, -3.640914465781471e+35, -3.4384183123816504e+35, -3.323444391361387e+35, -3.3002069361397204e+35, -3.3389434848528595e+35, -3.389311683423101e+35, -3.399565745364116e+35, -3.3356552438473276e+35, -3.1925476090376163e+35, -2.9942600403759607e+35, -2.7843440154834265e+35, -2.6107584979192128e+35, -2.5103166945094238e+35, -2.4978371037677495e+35, -2.562543311671631e+35, -2.671904548849696e+35, -2.7821505641963525e+35, -2.852734407361977e+35, -2.8595422531310247e+35, -2.8020151044890193e+35, -2.7024712835611056e+35, -2.5988649126880587e+35, -2.5333967871288085e+35, -2.5392695035104584e+35, -2.6283618364335336e+35, -2.7844533330803694e+35, -2.966772451474055e+35, -3.1240852153299685e+35, -3.213152150968412e+35, -3.213076510702336e+35, -3.130480962679199e+35, -2.9955648798126503e+35, -2.8516293553012813e+35, -2.7406909505144677e+35, -2.688579157015792e+35, -2.695270648810262e+35, -2.736006996105263e+35, -2.773385162986775e+35, -2.774360727625004e+35, -2.7241566914597683e+35, -2.6316676346133578e+35, -2.525197300367254e+35, -2.440814296901805e+35, -2.407676215853365e+35, -2.4358698982865808e+35, -2.512199489581533e+35, -2.6063373372767562e+35, -2.6840806544792302e+35, -2.7203665418341973e+35, -2.706440498343875e+35, -2.650907480301515e+35, -2.576740154775426e+35, -2.5143859820168666e+35, -2.4906426104226767e+35, -2.5168976863367503e+35, -2.5835259803462152e+35, -2.6645131584951614e+35, -2.7297983250422457e+35, -2.7581462244230507e+35, -2.7438055778639575e+35, -2.695161642144437e+35, -2.6291340825589534e+35, -2.565914286716902e+35, -2.524734376302357e+35, -2.519229630775704e+35, -2.5531931246826178e+35, -2.6190411201278633e+35, -2.6994465462209603e+35, -2.771325271478407e+35, -2.8123793465153617e+35, -2.8096237256732697e+35, -2.766074191250389e+35, -2.700810497686387e+35, -2.6408350677750305e+35, -2.6081041648786993e+35, -2.6087455471707807e+35, -2.631077341145884e+35, -2.653781011486725e+35, -2.658844976795883e+35, -2.6414033422281074e+35, -2.6119517918729495e+35, -2.5915599304789574e+35, -2.6033789605251438e+35, -2.6634369681638527e+35, -2.7726660328059582e+35, -2.9125680014088237e+35, -3.0481611591940532e+35, -3.13971128119832e+35, -3.1587454462873567e+35, -3.1004475584884932e+35, -2.9870084965029247e+35, -2.860611954821586e+35, -2.7676446072048074e+35, -2.738976632479152e+35, -2.7750077631813383e+35, -2.84466581582375e+35, -2.9005203483886495e+35, -2.9013761425073133e+35, -2.8298889675082635e+35, -2.6991808665572743e+35, -2.549238645614911e+35, -2.4336010499178597e+35, -2.396831227881686e+35, -2.4512506959135017e+35, -2.5681682991507098e+35, -2.6913782247380456e+35, -2.763849292496462e+35, -2.7508221256517044e+35, -2.650431462489204e+35, -2.492779667219443e+35, -2.329737398851255e+35, -2.2157547282075606e+35, -2.1839260819597773e+35]
tiempo_s_1emin5 = [0.0, 3.335640951999983e-06, 6.671281903999941e-06, 1.000692285600002e-05, 1.3342563808000148e-05, 1.6678204760000132e-05, 2.001384571199992e-05, 2.334948666399971e-05, 2.66851276159995e-05, 3.002076856799929e-05, 3.335640951999908e-05, 3.669205047199887e-05, 4.002769142399866e-05, 4.3363332375998447e-05, 4.6698973327998236e-05, 5.0034614279998025e-05, 5.3370255231997814e-05, 5.6705896183997604e-05, 6.004153713599739e-05, 6.337717808799814e-05, 6.671281903999928e-05, 7.004845999200043e-05, 7.338410094400157e-05, 7.671974189600272e-05, 8.005538284800386e-05, 8.3391023800005e-05, 8.672666475200615e-05, 9.00623057040073e-05, 9.339794665600844e-05, 9.673358760800958e-05, 0.00010006922856001073, 0.00010340486951201187, 0.00010674051046401302, 0.00011007615141601416, 0.0001134117923680153, 0.00011674743332001645, 0.0001200830742720176, 0.00012341871522401763, 0.00012675435617601606, 0.0001300899971280145, 0.00013342563808001293, 0.00013676127903201136, 0.0001400969199840098, 0.00014343256093600823, 0.00014676820188800667, 0.0001501038428400051, 0.00015343948379200353, 0.00015677512474400197, 0.0001601107656960004, 0.00016344640664799884, 0.00016678204759999727, 0.0001701176885519957, 0.00017345332950399414, 0.00017678897045599257, 0.000180124611407991, 0.00018346025235998944, 0.00018679589331198787, 0.0001901315342639863, 0.00019346717521598474, 0.00019680281616798318, 0.0002001384571199816, 0.00020347409807198005, 0.00020680973902397848, 0.0002101453799759769, 0.00021348102092797535, 0.00021681666187997378, 0.00022015230283197222, 0.00022348794378397065, 0.00022682358473596908, 0.00023015922568796752, 0.00023349486663996595, 0.00023683050759196439, 0.00024016614854396282, 0.00024350178949596125, 0.0002468374304479553, 0.0002501730713999483, 0.0002535087123519413, 0.00025684435330393434, 0.00026017999425592735, 0.00026351563520792036, 0.0002668512761599134, 0.0002701869171119064, 0.0002735225580638994, 0.0002768581990158924, 0.0002801938399678854, 0.00028352948091987844, 0.00028686512187187145, 0.00029020076282386447, 0.0002935364037758575, 0.0002968720447278505, 0.0003002076856798435, 0.0003035433266318365, 0.00030687896758382953, 0.00031021460853582254, 0.00031355024948781556, 0.00031688589043980857, 0.0003202215313918016, 0.0003235571723437946, 0.0003268928132957876, 0.0003302284542477806, 0.00033356409519977364, 0.00033689973615176665, 0.00034023537710375966, 0.0003435710180557527, 0.0003469066590077457, 0.0003502422999597387, 0.0003535779409117317, 0.0003569135818637247, 0.00036024922281571774, 0.00036358486376771075, 0.00036692050471970377, 0.0003702561456716968, 0.0003735917866236898, 0.0003769274275756828, 0.0003802630685276758, 0.00038359870947966883, 0.00038693435043166184, 0.00039026999138365486, 0.00039360563233564787, 0.0003969412732876409, 0.0004002769142396339, 0.0004036125551916269, 0.0004069481961436199, 0.00041028383709561293, 0.00041361947804760595, 0.00041695511899959896, 0.000420290759951592, 0.000423626400903585, 0.000426962041855578, 0.000430297682807571, 0.000433633323759564, 0.00043696896471155704, 0.00044030460566355005, 0.00044364024661554307, 0.0004469758875675361, 0.0004503115285195291, 0.0004536471694715221, 0.0004569828104235151, 0.00046031845137550813, 0.00046365409232750114, 0.00046698973327949416, 0.00047032537423148717, 0.0004736610151834802, 0.0004769966561354732, 0.0004803322970874662, 0.0004836679380394592, 0.00048700357899145223, 0.0004903392199434453, 0.0004936748608954383, 0.0004970105018474313, 0.0005003461427994243, 0.0005036817837514174, 0.0005070174247034104, 0.0005103530656554034, 0.0005136887066073964, 0.0005170243475593894, 0.0005203599885113824, 0.0005236956294633754, 0.0005270312704153684, 0.0005303669113673615, 0.0005337025523193545, 0.0005370381932713475, 0.0005403738342233405, 0.0005437094751753335, 0.0005470451161273265, 0.0005503807570793195, 0.0005537163980313125, 0.0005570520389833056, 0.0005603876799352986, 0.0005637233208872916, 0.0005670589618392846, 0.0005703946027912776, 0.0005737302437432706, 0.0005770658846952636, 0.0005804015256472567, 0.0005837371665992497, 0.0005870728075512427, 0.0005904084485032357, 0.0005937440894552287, 0.0005970797304072217, 0.0006004153713592147, 0.0006037510123112077, 0.0006070866532632008, 0.0006104222942151938, 0.0006137579351671868, 0.0006170935761191798, 0.0006204292170711728, 0.0006237648580231658, 0.0006271004989751588, 0.0006304361399271518, 0.0006337717808791449, 0.0006371074218311379, 0.0006404430627831309, 0.0006437787037351239, 0.0006471143446871169, 0.0006504499856391099, 0.0006537856265911029, 0.000657121267543096, 0.000660456908495089, 0.000663792549447082, 0.000667128190399075, 0.000670463831351068, 0.000673799472303061, 0.000677135113255054, 0.000680470754207047, 0.0006838063951590401, 0.0006871420361110331, 0.0006904776770630261, 0.0006938133180150191, 0.0006971489589670121, 0.0007004845999190051, 0.0007038202408709981, 0.0007071558818229911, 0.0007104915227749842, 0.0007138271637269772, 0.0007171628046789702, 0.0007204984456309632, 0.0007238340865829562, 0.0007271697275349492, 0.0007305053684869422, 0.0007338410094389353, 0.0007371766503909283, 0.0007405122913429213, 0.0007438479322949143, 0.0007471835732469073, 0.0007505192141989003, 0.0007538548551508933, 0.0007571904961028863, 0.0007605261370548794, 0.0007638617780068724, 0.0007671974189588654, 0.0007705330599108584, 0.0007738687008628514, 0.0007772043418148444, 0.0007805399827668374, 0.0007838756237188304, 0.0007872112646708235, 0.0007905469056228165, 0.0007938825465748095, 0.0007972181875268025, 0.0008005538284787955, 0.0008038894694307885, 0.0008072251103827815, 0.0008105607513347746, 0.0008138963922867676, 0.0008172320332387606, 0.0008205676741907536, 0.0008239033151427466, 0.0008272389560947396, 0.0008305745970467326, 0.0008339102379987256, 0.0008372458789507187, 0.0008405815199027117, 0.0008439171608547047, 0.0008472528018066977, 0.0008505884427586907, 0.0008539240837106837, 0.0008572597246626767, 0.0008605953656146697, 0.0008639310065666628, 0.0008672666475186558, 0.0008706022884706488, 0.0008739379294226418, 0.0008772735703746348, 0.0008806092113266278, 0.0008839448522786208, 0.0008872804932306139, 0.0008906161341826069, 0.0008939517751345999, 0.0008972874160865929, 0.0009006230570385859, 0.0009039586979905789, 0.0009072943389425719, 0.0009106299798945649, 0.000913965620846558, 0.000917301261798551, 0.000920636902750544, 0.000923972543702537, 0.00092730818465453, 0.000930643825606523, 0.000933979466558516, 0.000937315107510509, 0.0009406507484625021, 0.0009439863894144951, 0.0009473220303664881, 0.0009506576713184811, 0.0009539933122704741, 0.0009573289532224671, 0.0009606645941744601, 0.0009640002351264532, 0.0009673358760784462, 0.0009706715170304392, 0.0009740071579824322, 0.0009773427989344253, 0.0009806784398864183, 0.0009840140808384113, 0.0009873497217904044, 0.0009906853627423974, 0.0009940210036943904, 0.0009973566446463834, 0.0010006922855983764, 0.0010040279265503694, 0.0010073635675023624, 0.0010106992084543554, 0.0010140348494063485, 0.0010173704903583415, 0.0010207061313103345, 0.0010240417722623275, 0.0010273774132143205, 0.0010307130541663135, 0.0010340486951183065, 0.0010373843360702995, 0.0010407199770222926, 0.0010440556179742856, 0.0010473912589262786, 0.0010507268998782716, 0.0010540625408302646, 0.0010573981817822576, 0.0010607338227342506, 0.0010640694636862437, 0.0010674051046382367, 0.0010707407455902297, 0.0010740763865422227, 0.0010774120274942157, 0.0010807476684462087, 0.0010840833093982017, 0.0010874189503501947, 0.0010907545913021878, 0.0010940902322541808, 0.0010974258732061738, 0.0011007615141581668, 0.0011040971551101598, 0.0011074327960621528, 0.0011107684370141458, 0.0011141040779661388, 0.0011174397189181319, 0.0011207753598701249, 0.0011241110008221179, 0.001127446641774111, 0.001130782282726104, 0.001134117923678097, 0.00113745356463009, 0.001140789205582083, 0.001144124846534076, 0.001147460487486069, 0.001150796128438062, 0.001154131769390055, 0.001157467410342048, 0.001160803051294041, 0.001164138692246034, 0.001167474333198027, 0.00117080997415002, 0.001174145615102013, 0.001177481256054006, 0.001180816897005999, 0.0011841525379579921, 0.0011874881789099851, 0.0011908238198619781, 0.0011941594608139712, 0.0011974951017659642, 0.0012008307427179572, 0.0012041663836699502, 0.0012075020246219432, 0.0012108376655739362, 0.0012141733065259292, 0.0012175089474779222, 0.0012208445884299153, 0.0012241802293819083, 0.0012275158703339013, 0.0012308515112858943, 0.0012341871522378873, 0.0012375227931898803, 0.0012408584341418733, 0.0012441940750938664, 0.0012475297160458594, 0.0012508653569978524, 0.0012542009979498454, 0.0012575366389018384, 0.0012608722798538314, 0.0012642079208058244, 0.0012675435617578174, 0.0012708792027098105, 0.0012742148436618035, 0.0012775504846137965, 0.0012808861255657895, 0.0012842217665177825, 0.0012875574074697755, 0.0012908930484217685, 0.0012942286893737615, 0.0012975643303257546, 0.0013008999712777476, 0.0013042356122297406, 0.0013075712531817336, 0.0013109068941337266, 0.0013142425350857196, 0.0013175781760377126, 0.0013209138169897057, 0.0013242494579416987, 0.0013275850988936917, 0.0013309207398456847, 0.0013342563807976777, 0.0013375920217496707, 0.0013409276627016637, 0.0013442633036536567, 0.0013475989446056498, 0.0013509345855576428, 0.0013542702265096358, 0.0013576058674616288, 0.0013609415084136218, 0.0013642771493656148, 0.0013676127903176078, 0.0013709484312696008, 0.0013742840722215939, 0.0013776197131735869, 0.0013809553541255799, 0.001384290995077573, 0.001387626636029566, 0.001390962276981559, 0.001394297917933552, 0.001397633558885545, 0.001400969199837538, 0.001404304840789531, 0.001407640481741524, 0.001410976122693517, 0.00141431176364551, 0.001417647404597503, 0.001420983045549496, 0.001424318686501489, 0.001427654327453482, 0.001430989968405475, 0.001434325609357468, 0.0014376612503094611, 0.0014409968912614541, 0.0014443325322134471, 0.0014476681731654401, 0.0014510038141174332, 0.0014543394550694262, 0.0014576750960214192, 0.0014610107369734122, 0.0014643463779254052, 0.0014676820188773982, 0.0014710176598293912, 0.0014743533007813843, 0.0014776889417333773, 0.0014810245826853703, 0.0014843602236373633, 0.0014876958645893563, 0.0014910315055413493, 0.0014943671464933423, 0.0014977027874453353, 0.0015010384283973284, 0.0015043740693493214, 0.0015077097103013144, 0.0015110453512533074, 0.0015143809922053004, 0.0015177166331572934, 0.0015210522741092864, 0.0015243879150612794, 0.0015277235560132725, 0.0015310591969652655, 0.0015343948379172585, 0.0015377304788692515, 0.0015410661198212445, 0.0015444017607732375, 0.0015477374017252305, 0.0015510730426772236, 0.0015544086836292166, 0.0015577443245812096, 0.0015610799655332026, 0.0015644156064851956, 0.0015677512474371886, 0.0015710868883891816, 0.0015744225293411746, 0.0015777581702931677, 0.0015810938112451607, 0.0015844294521971537, 0.0015877650931491467, 0.0015911007341011397, 0.0015944363750531327, 0.0015977720160051257, 0.0016011076569571187, 0.0016044432979091118, 0.0016077789388611048, 0.0016111145798130978, 0.0016144502207650908, 0.0016177858617170838, 0.0016211215026690768, 0.0016244571436210698, 0.0016277927845730629, 0.0016311284255250559, 0.0016344640664770489, 0.0016377997074290419, 0.001641135348381035, 0.001644470989333028, 0.001647806630285021, 0.001651142271237014, 0.001654477912189007, 0.001657813553141, 0.001661149194092993, 0.001664484835044986, 0.001667820475996979, 0.001671156116948972, 0.001674491757900965, 0.001677827398852958, 0.001681163039804951, 0.001684498680756944, 0.001687834321708937, 0.00169116996266093, 0.0016945056036129231, 0.0016978412445649161, 0.0017011768855169091, 0.0017045125264689022, 0.0017078481674208952, 0.0017111838083728882, 0.0017145194493248812, 0.0017178550902768742, 0.0017211907312288672, 0.0017245263721808602, 0.0017278620131328532, 0.0017311976540848463, 0.0017345332950368393, 0.0017378689359888323, 0.0017412045769408253, 0.0017445402178928183, 0.0017478758588448113, 0.0017512114997968043, 0.0017545471407487973, 0.0017578827817007904, 0.0017612184226527834, 0.0017645540636047764, 0.0017678897045567694, 0.0017712253455087624, 0.0017745609864607554, 0.0017778966274127484, 0.0017812322683647415, 0.0017845679093167345, 0.0017879035502687275, 0.0017912391912207205, 0.0017945748321727135, 0.0017979104731247065, 0.0018012461140766995, 0.0018045817550286925, 0.0018079173959806856, 0.0018112530369326786, 0.0018145886778846716, 0.0018179243188366646, 0.0018212599597886576, 0.0018245956007406506, 0.0018279312416926436, 0.0018312668826446366, 0.0018346025235966297, 0.0018379381645486227, 0.0018412738055006157, 0.0018446094464526087, 0.0018479450874046017, 0.0018512807283565947, 0.0018546163693085877, 0.0018579520102605808, 0.0018612876512125738, 0.0018646232921645668, 0.0018679589331165598, 0.0018712945740685528, 0.0018746302150205458, 0.0018779658559725388, 0.0018813014969245318, 0.0018846371378765249, 0.0018879727788285179, 0.0018913084197805109, 0.001894644060732504, 0.001897979701684497, 0.00190131534263649, 0.001904650983588483, 0.001907986624540476, 0.001911322265492469, 0.001914657906444462, 0.001917993547396455, 0.001921329188348448, 0.001924664829300441, 0.001928000470252434, 0.001931336111204427, 0.00193467175215642, 0.001938007393108413, 0.001941343034060406, 0.001944678675012399, 0.001948014315964392, 0.0019513499569163851, 0.0019546855978683783, 0.0019580212388203714, 0.0019613568797723644, 0.0019646925207243574, 0.0019680281616763504, 0.0019713638026283434, 0.0019746994435803364, 0.0019780350845323294, 0.0019813707254843224, 0.0019847063664363155, 0.0019880420073883085, 0.0019913776483403015, 0.0019947132892922945, 0.0019980489302442875, 0.0020013845711962805, 0.0020047202121482735, 0.0020080558531002666, 0.0020113914940522596, 0.0020147271350042526, 0.0020180627759562456, 0.0020213984169082386, 0.0020247340578602316, 0.0020280696988122246, 0.0020314053397642176, 0.0020347409807162107, 0.0020380766216682037, 0.0020414122626201967, 0.0020447479035721897, 0.0020480835445241827, 0.0020514191854761757, 0.0020547548264281687, 0.0020580904673801617, 0.0020614261083321548, 0.0020647617492841478, 0.002068097390236141, 0.002071433031188134, 0.002074768672140127, 0.00207810431309212, 0.002081439954044113, 0.002084775594996106, 0.002088111235948099, 0.002091446876900092, 0.002094782517852085, 0.002098118158804078, 0.002101453799756071, 0.002104789440708064, 0.002108125081660057, 0.00211146072261205, 0.002114796363564043, 0.002118132004516036, 0.002121467645468029, 0.002124803286420022, 0.002128138927372015, 0.002131474568324008, 0.002134810209276001, 0.002138145850227994, 0.002141481491179987, 0.00214481713213198, 0.002148152773083973, 0.002151488414035966, 0.002154824054987959, 0.002158159695939952, 0.002161495336891945, 0.002164830977843938, 0.002168166618795931, 0.002171502259747924, 0.002174837900699917, 0.0021781735416519102, 0.0021815091826039032, 0.0021848448235558962, 0.0021881804645078893, 0.0021915161054598823, 0.0021948517464118753, 0.0021981873873638683, 0.0022015230283158613, 0.0022048586692678543, 0.0022081943102198473, 0.0022115299511718403, 0.0022148655921238334, 0.0022182012330758264, 0.0022215368740278194, 0.0022248725149798124, 0.0022282081559318054, 0.0022315437968837984, 0.0022348794378357914, 0.0022382150787877845, 0.0022415507197397775, 0.0022448863606917705, 0.0022482220016437635, 0.0022515576425957565, 0.0022548932835477495, 0.0022582289244997425, 0.0022615645654517355, 0.0022649002064037286, 0.0022682358473557216, 0.0022715714883077146, 0.0022749071292597076, 0.0022782427702117006, 0.0022815784111636936, 0.0022849140521156866, 0.0022882496930676796, 0.0022915853340196727, 0.0022949209749716657, 0.0022982566159236587, 0.0023015922568756517, 0.0023049278978276447, 0.0023082635387796377, 0.0023115991797316307, 0.0023149348206836238, 0.0023182704616356168, 0.0023216061025876098, 0.002324941743539603, 0.002328277384491596, 0.002331613025443589, 0.002334948666395582, 0.002338284307347575, 0.002341619948299568, 0.002344955589251561, 0.002348291230203554, 0.002351626871155547, 0.00235496251210754, 0.002358298153059533, 0.002361633794011526, 0.002364969434963519, 0.002368305075915512, 0.002371640716867505, 0.002374976357819498, 0.002378311998771491, 0.002381647639723484, 0.002384983280675477, 0.00238831892162747, 0.002391654562579463, 0.002394990203531456, 0.002398325844483449, 0.002401661485435442, 0.002404997126387435, 0.002408332767339428, 0.002411668408291421, 0.002415004049243414, 0.002418339690195407, 0.0024216753311474, 0.002425010972099393, 0.002428346613051386, 0.002431682254003379, 0.0024350178949553722, 0.0024383535359073652, 0.0024416891768593582, 0.0024450248178113513, 0.0024483604587633443, 0.0024516960997153373, 0.0024550317406673303, 0.0024583673816193233, 0.0024617030225713163, 0.0024650386635233093, 0.0024683743044753024, 0.0024717099454272954, 0.0024750455863792884, 0.0024783812273312814, 0.0024817168682832744, 0.0024850525092352674, 0.0024883881501872604, 0.0024917237911392534, 0.0024950594320912465, 0.0024983950730432395, 0.0025017307139952325, 0.0025050663549472255, 0.0025084019958992185, 0.0025117376368512115, 0.0025150732778032045, 0.0025184089187551975, 0.0025217445597071906, 0.0025250802006591836, 0.0025284158416111766, 0.0025317514825631696, 0.0025350871235151626, 0.0025384227644671556, 0.0025417584054191486, 0.0025450940463711417, 0.0025484296873231347, 0.0025517653282751277, 0.0025551009692271207, 0.0025584366101791137, 0.0025617722511311067, 0.0025651078920830997, 0.0025684435330350927, 0.0025717791739870858, 0.0025751148149390788, 0.0025784504558910718, 0.002581786096843065, 0.002585121737795058, 0.002588457378747051, 0.002591793019699044, 0.002595128660651037, 0.00259846430160303, 0.002601799942555023, 0.002605135583507016, 0.002608471224459009, 0.002611806865411002, 0.002615142506362995, 0.002618478147314988, 0.002621813788266981, 0.002625149429218974, 0.002628485070170967, 0.00263182071112296, 0.002635156352074953, 0.002638491993026946, 0.002641827633978939, 0.002645163274930932, 0.002648498915882925, 0.002651834556834918, 0.002655170197786911, 0.002658505838738904, 0.002661841479690897, 0.00266517712064289, 0.002668512761594883, 0.002671848402546876, 0.002675184043498869, 0.002678519684450862, 0.002681855325402855, 0.002685190966354848, 0.002688526607306841, 0.0026918622482588342, 0.0026951978892108272, 0.0026985335301628203, 0.0027018691711148133, 0.0027052048120668063, 0.0027085404530187993, 0.0027118760939707923, 0.0027152117349227853, 0.0027185473758747783, 0.0027218830168267713, 0.0027252186577787644, 0.0027285542987307574, 0.0027318899396827504, 0.0027352255806347434, 0.0027385612215867364, 0.0027418968625387294, 0.0027452325034907224, 0.0027485681444427154, 0.0027519037853947085, 0.0027552394263467015, 0.0027585750672986945, 0.0027619107082506875, 0.0027652463492026805, 0.0027685819901546735, 0.0027719176311066665, 0.0027752532720586595, 0.0027785889130106526, 0.0027819245539626456, 0.0027852601949146386, 0.0027885958358666316, 0.0027919314768186246, 0.0027952671177706176, 0.0027986027587226106, 0.0028019383996746037, 0.0028052740406265967, 0.0028086096815785897, 0.0028119453225305827, 0.0028152809634825757, 0.0028186166044345687, 0.0028219522453865617, 0.0028252878863385547, 0.0028286235272905478, 0.0028319591682425408, 0.002835294809194534, 0.002838630450146527, 0.00284196609109852, 0.002845301732050513, 0.002848637373002506, 0.002851973013954499, 0.002855308654906492, 0.002858644295858485, 0.002861979936810478, 0.002865315577762471, 0.002868651218714464, 0.002871986859666457, 0.00287532250061845, 0.002878658141570443, 0.002881993782522436, 0.002885329423474429, 0.002888665064426422, 0.002892000705378415, 0.002895336346330408, 0.002898671987282401, 0.002902007628234394, 0.002905343269186387, 0.00290867891013838, 0.002912014551090373, 0.002915350192042366, 0.002918685832994359, 0.002922021473946352, 0.002925357114898345, 0.002928692755850338, 0.002932028396802331, 0.002935364037754324, 0.002938699678706317, 0.00294203531965831, 0.002945370960610303, 0.0029487066015622962, 0.0029520422425142892, 0.0029553778834662823, 0.0029587135244182753, 0.0029620491653702683, 0.0029653848063222613, 0.0029687204472742543, 0.0029720560882262473, 0.0029753917291782403, 0.0029787273701302333, 0.0029820630110822264, 0.0029853986520342194, 0.0029887342929862124, 0.0029920699339382054, 0.0029954055748901984, 0.0029987412158421914, 0.0030020768567941844, 0.0030054124977461774, 0.0030087481386981705, 0.0030120837796501635, 0.0030154194206021565, 0.0030187550615541495, 0.0030220907025061425, 0.0030254263434581355, 0.0030287619844101285, 0.0030320976253621216, 0.0030354332663141146, 0.0030387689072661076, 0.0030421045482181006, 0.0030454401891700936, 0.0030487758301220866, 0.0030521114710740796, 0.0030554471120260726, 0.0030587827529780657, 0.0030621183939300587, 0.0030654540348820517, 0.0030687896758340447, 0.0030721253167860377, 0.0030754609577380307, 0.0030787965986900237, 0.0030821322396420167, 0.0030854678805940098, 0.0030888035215460028, 0.003092139162497996, 0.003095474803449989, 0.003098810444401982, 0.003102146085353975, 0.003105481726305968, 0.003108817367257961, 0.003112153008209954, 0.003115488649161947, 0.00311882429011394, 0.003122159931065933, 0.003125495572017926, 0.003128831212969919, 0.003132166853921912, 0.003135502494873905, 0.003138838135825898, 0.003142173776777891, 0.003145509417729884, 0.003148845058681877, 0.00315218069963387, 0.003155516340585863, 0.003158851981537856, 0.003162187622489849, 0.003165523263441842, 0.003168858904393835, 0.003172194545345828, 0.003175530186297821, 0.003178865827249814, 0.003182201468201807, 0.0031855371091538, 0.003188872750105793, 0.003192208391057786, 0.003195544032009779, 0.003198879672961772, 0.0032022153139137652, 0.0032055509548657582, 0.0032088865958177512, 0.0032122222367697443, 0.0032155578777217373, 0.0032188935186737303, 0.0032222291596257233, 0.0032255648005777163, 0.0032289004415297093, 0.0032322360824817023, 0.0032355717234336953, 0.0032389073643856884, 0.0032422430053376814, 0.0032455786462896744, 0.0032489142872416674, 0.0032522499281936604, 0.0032555855691456534, 0.0032589212100976464, 0.0032622568510496395, 0.0032655924920016325, 0.0032689281329536255, 0.0032722637739056185, 0.0032755994148576115, 0.0032789350558096045, 0.0032822706967615975, 0.0032856063377135905, 0.0032889419786655836, 0.0032922776196175766, 0.0032956132605695696, 0.0032989489015215626, 0.0033022845424735556, 0.0033056201834255486, 0.0033089558243775416, 0.0033122914653295346, 0.0033156271062815277, 0.0033189627472335207, 0.0033222983881855137, 0.0033256340291375067, 0.0033289696700894997, 0.0033323053110414927, 0.0033356409519934857, 0.0033389765929454788, 0.0033423122338974718, 0.0033456478748494648, 0.003348983515801458, 0.003352319156753451, 0.003355654797705444, 0.003358990438657437, 0.00336232607960943, 0.003365661720561423, 0.003368997361513416, 0.003372333002465409, 0.003375668643417402, 0.003379004284369395, 0.003382339925321388, 0.003385675566273381, 0.003389011207225374, 0.003392346848177367, 0.00339568248912936, 0.003399018130081353, 0.003402353771033346, 0.003405689411985339, 0.003409025052937332, 0.003412360693889325, 0.003415696334841318, 0.003419031975793311, 0.003422367616745304, 0.003425703257697297, 0.00342903889864929, 0.003432374539601283, 0.003435710180553276, 0.003439045821505269, 0.003442381462457262, 0.003445717103409255, 0.003449052744361248, 0.003452388385313241, 0.003455724026265234, 0.0034590596672172272, 0.0034623953081692202, 0.0034657309491212132, 0.0034690665900732063, 0.0034724022310251993, 0.0034757378719771923, 0.0034790735129291853, 0.0034824091538811783, 0.0034857447948331713, 0.0034890804357851643, 0.0034924160767371574, 0.0034957517176891504, 0.0034990873586411434, 0.0035024229995931364, 0.0035057586405451294, 0.0035090942814971224, 0.0035124299224491154, 0.0035157655634011084, 0.0035191012043531015, 0.0035224368453050945, 0.0035257724862570875, 0.0035291081272090805, 0.0035324437681610735, 0.0035357794091130665, 0.0035391150500650595, 0.0035424506910170525, 0.0035457863319690456, 0.0035491219729210386, 0.0035524576138730316, 0.0035557932548250246, 0.0035591288957770176, 0.0035624645367290106, 0.0035658001776810036, 0.0035691358186329966, 0.0035724714595849897, 0.0035758071005369827, 0.0035791427414889757, 0.0035824783824409687, 0.0035858140233929617, 0.0035891496643449547, 0.0035924853052969477, 0.0035958209462489408, 0.0035991565872009338, 0.0036024922281529268, 0.00360582786910492, 0.003609163510056913, 0.003612499151008906, 0.003615834791960899, 0.003619170432912892, 0.003622506073864885, 0.003625841714816878, 0.003629177355768871, 0.003632512996720864, 0.003635848637672857, 0.00363918427862485, 0.003642519919576843, 0.003645855560528836, 0.003649191201480829, 0.003652526842432822, 0.003655862483384815, 0.003659198124336808, 0.003662533765288801, 0.003665869406240794, 0.003669205047192787, 0.00367254068814478, 0.003675876329096773, 0.003679211970048766, 0.003682547611000759, 0.003685883251952752, 0.003689218892904745, 0.003692554533856738, 0.003695890174808731, 0.003699225815760724, 0.003702561456712717, 0.00370589709766471, 0.003709232738616703, 0.003712568379568696, 0.0037159040205206892, 0.0037192396614726822, 0.0037225753024246752, 0.0037259109433766683, 0.0037292465843286613, 0.0037325822252806543, 0.0037359178662326473, 0.0037392535071846403, 0.0037425891481366333, 0.0037459247890886263, 0.0037492604300406194, 0.0037525960709926124, 0.0037559317119446054, 0.0037592673528965984, 0.0037626029938485914, 0.0037659386348005844, 0.0037692742757525774, 0.0037726099167045704, 0.0037759455576565635, 0.0037792811986085565, 0.0037826168395605495, 0.0037859524805125425, 0.0037892881214645355, 0.0037926237624165285, 0.0037959594033685215, 0.0037992950443205145, 0.0038026306852725076, 0.0038059663262245006, 0.0038093019671764936, 0.0038126376081284866, 0.0038159732490804796, 0.0038193088900324726, 0.0038226445309844656, 0.0038259801719364587, 0.0038293158128884517, 0.0038326514538404447, 0.0038359870947924377, 0.0038393227357444307, 0.0038426583766964237, 0.0038459940176484167, 0.0038493296586004097, 0.0038526652995524028, 0.0038560009405043958, 0.0038593365814563888, 0.003862672222408382, 0.003866007863360375, 0.003869343504312368, 0.003872679145264361, 0.003876014786216354, 0.003879350427168347, 0.00388268606812034, 0.003886021709072333, 0.003889357350024326, 0.003892692990976319, 0.003896028631928312, 0.003899364272880305, 0.003902699913832298, 0.003906035554784291, 0.003909371195736284, 0.003912706836688277, 0.00391604247764027, 0.003919378118592263, 0.003922713759544256, 0.003926049400496249, 0.003929385041448242, 0.003932720682400235, 0.003936056323352228, 0.003939391964304221, 0.003942727605256214, 0.003946063246208207, 0.0039493988871602, 0.003952734528112193, 0.003956070169064186, 0.003959405810016179, 0.003962741450968172, 0.003966077091920165, 0.003969412732872158, 0.003972748373824151, 0.003976084014776144, 0.003979419655728137, 0.00398275529668013, 0.003986090937632123, 0.003989426578584116, 0.003992762219536109, 0.003996097860488102, 0.003999433501440095, 0.004002769142392088, 0.004034791295531221, 0.004066813448670354, 0.004098835601809487, 0.00413085775494862, 0.004162879908087753, 0.004194902061226886, 0.004226924214366019, 0.004258946367505152, 0.004290968520644285, 0.004322990673783418, 0.0043550128269225505, 0.0043870349800616834, 0.004419057133200816, 0.004451079286339949, 0.004483101439479082, 0.004515123592618215, 0.004547145745757348, 0.004579167898896481, 0.004611190052035614, 0.004643212205174747, 0.00467523435831388, 0.004707256511453013, 0.004739278664592146, 0.0047713008177312785, 0.0048033229708704115, 0.004835345124009544, 0.004867367277148677, 0.00489938943028781, 0.004931411583426943, 0.004963433736566076, 0.004995455889705209, 0.005027478042844342, 0.005059500195983475, 0.005091522349122608, 0.005123544502261741, 0.005155566655400874, 0.005187588808540007, 0.0052196109616791395, 0.005251633114818272, 0.005283655267957405, 0.005315677421096538, 0.005347699574235671, 0.005379721727374804, 0.005411743880513937, 0.00544376603365307, 0.005475788186792203, 0.005507810339931336, 0.005539832493070469, 0.005571854646209602, 0.005603876799348735, 0.0056358989524878675, 0.0056679211056270004, 0.005699943258766133, 0.005731965411905266, 0.005763987565044399, 0.005796009718183532, 0.005828031871322665, 0.005860054024461798, 0.005892076177600931, 0.005924098330740064, 0.005956120483879197, 0.00598814263701833, 0.006020164790157463, 0.0060521869432965955, 0.0060842090964357285, 0.006116231249574861, 0.006148253402713994, 0.006180275555853127, 0.00621229770899226, 0.006244319862131393, 0.006276342015270526, 0.006308364168409659, 0.006340386321548792, 0.006372408474687925, 0.006404430627827058, 0.006436452780966191, 0.006468474934105324, 0.0065004970872444565, 0.006532519240383589, 0.006564541393522722, 0.006596563546661855, 0.006628585699800988, 0.006660607852940121, 0.006692630006079254, 0.006724652159218387, 0.00675667431235752, 0.006788696465496653, 0.006820718618635786, 0.006852740771774919, 0.006884762924914052, 0.0069167850780531845, 0.0069488072311923174, 0.00698082938433145, 0.007012851537470583, 0.007044873690609716, 0.007076895843748849, 0.007108917996887982, 0.007140940150027115, 0.007172962303166248, 0.007204984456305381, 0.007237006609444514, 0.007269028762583647, 0.00730105091572278, 0.0073330730688619125, 0.0073650952220010455, 0.007397117375140178, 0.007429139528279311, 0.007461161681418444, 0.007493183834557577, 0.00752520598769671, 0.007557228140835843, 0.007589250293974976, 0.007621272447114109, 0.007653294600253242, 0.007685316753392375, 0.007717338906531508, 0.007749361059670641, 0.0077813832128097735, 0.007813405365948907, 0.00784542751908804, 0.007877449672227173, 0.007909471825366306, 0.007941493978505439, 0.007973516131644572, 0.008005538284783705, 0.008037560437922838, 0.00806958259106197, 0.008101604744201104, 0.008133626897340237, 0.00816564905047937, 0.008197671203618502, 0.008229693356757635, 0.008261715509896768, 0.008293737663035901, 0.008325759816175034, 0.008357781969314167, 0.0083898041224533, 0.008421826275592433, 0.008453848428731566, 0.008485870581870699, 0.008517892735009832, 0.008549914888148965, 0.008581937041288097, 0.00861395919442723, 0.008645981347566363, 0.008678003500705496, 0.00871002565384463, 0.008742047806983762, 0.008774069960122895, 0.008806092113262028, 0.008838114266401161, 0.008870136419540294, 0.008902158572679427, 0.00893418072581856, 0.008966202878957693, 0.008998225032096826, 0.009030247185235958, 0.009062269338375091, 0.009094291491514224, 0.009126313644653357, 0.00915833579779249, 0.009190357950931623, 0.009222380104070756, 0.009254402257209889, 0.009286424410349022, 0.009318446563488155, 0.009350468716627288, 0.00938249086976642, 0.009414513022905554, 0.009446535176044686, 0.00947855732918382, 0.009510579482322952, 0.009542601635462085, 0.009574623788601218, 0.009606645941740351, 0.009638668094879484, 0.009670690248018617, 0.00970271240115775, 0.009734734554296883, 0.009766756707436016, 0.009798778860575149, 0.009830801013714282, 0.009862823166853414, 0.009894845319992547, 0.00992686747313168, 0.009958889626270813, 0.009990911779409946, 0.010022933932549079, 0.010054956085688212, 0.010086978238827345, 0.010119000391966478, 0.01015102254510561, 0.010183044698244744, 0.010215066851383877, 0.01024708900452301, 0.010279111157662143, 0.010311133310801275, 0.010343155463940408, 0.010375177617079541, 0.010407199770218674, 0.010439221923357807, 0.01047124407649694, 0.010503266229636073, 0.010535288382775206, 0.010567310535914339, 0.010599332689053472, 0.010631354842192605, 0.010663376995331738, 0.01069539914847087, 0.010727421301610003, 0.010759443454749136, 0.01079146560788827, 0.010823487761027402, 0.010855509914166535, 0.010887532067305668, 0.010919554220444801, 0.010951576373583934, 0.010983598526723067, 0.0110156206798622, 0.011047642833001333, 0.011079664986140466, 0.011111687139279599, 0.011143709292418731, 0.011175731445557864, 0.011207753598696997, 0.01123977575183613, 0.011271797904975263, 0.011303820058114396, 0.011335842211253529, 0.011367864364392662, 0.011399886517531795, 0.011431908670670928, 0.01146393082381006, 0.011495952976949194, 0.011527975130088327, 0.01155999728322746, 0.011592019436366592, 0.011624041589505725, 0.011656063742644858, 0.011688085895783991, 0.011720108048923124, 0.011752130202062257, 0.01178415235520139, 0.011816174508340523, 0.011848196661479656, 0.011880218814618789, 0.011912240967757922, 0.011944263120897055, 0.011976285274036188, 0.01200830742717532]

net_absorption_emission_protons_inv_s_inv_cm3_1emin4 = [0.0, -1.829073682953341e+35, -1.5548214130002747e+35, -1.3784956449519795e+35, -1.27041771826496e+35, -1.2010111377809486e+35, -1.1543190271670717e+35, -1.1216587489248443e+35, -1.0980695358944582e+35, -1.0805766929593541e+35, -1.0673307983992603e+35, -1.0571910912730559e+35, -1.0496372003495227e+35, -1.0452786842578144e+35, -1.0483754591393808e+35, -1.0771470507658363e+35, -1.2047630279964377e+35, -1.7192432212043285e+35, -3.708448664342149e+35, -1.0651632470880761e+36, -2.709503546877272e+36, -3.610586254370403e+36, -2.8527051064892924e+36, -2.1085630235014963e+36, -1.6020254617120875e+36, -1.3021992234470211e+36, -1.107057750739173e+36, -1.0690822568609194e+36, -1.2651223259146615e+36, -1.7177797085012929e+36, -1.9453524815573112e+36, -1.7128411343794652e+36, -1.3773790393988972e+36, -1.1996926281949024e+36, -1.1353673750541419e+36, -1.175182999111856e+36, -1.2937964862917748e+36, -1.4325647461698934e+36, -1.316157417382951e+36, -1.1235916691887836e+36, -9.500059719311115e+35, -8.621154934025326e+35, -7.820171768544418e+35, -7.234553367005493e+35, -6.878068185975513e+35, -6.543001550744575e+35, -7.35859009250238e+35, -8.155797616921128e+35, -8.901528550019706e+35, -1.0257083356011592e+36, -1.1357548090533861e+36, -1.1204761592511549e+36, -1.0833142104948538e+36, -9.205392547722045e+35, -7.996698322531472e+35, -7.306683681754678e+35, -6.5064254917462275e+35, -6.1246627272163844e+35, -6.2890388332329175e+35, -7.205933349087429e+35, -8.292309855804685e+35, -9.195393308233671e+35, -1.0218851588259525e+36, -1.0844086306010617e+36, -1.0713231797951499e+36, -1.0136561425106663e+36, -9.73779936823035e+35, -9.478027687782372e+35, -8.622551389736692e+35, -7.918658409271644e+35, -8.033120899921421e+35, -8.451684728764278e+35, -9.295555414138546e+35, -1.0507358859115971e+36, -1.1385981136188046e+36, -1.2119011355927736e+36, -1.1119700104673817e+36, -9.848085859522315e+35, -8.982182707776228e+35, -7.989581369477587e+35, -7.800567300086198e+35, -8.099778080006782e+35, -8.248590580280258e+35, -8.255388984235786e+35, -8.140778942321102e+35, -7.76780237406012e+35, -7.012314789910115e+35, -5.905611770960824e+35, -5.11349340599461e+35, -4.726566047276193e+35, -4.516139133614444e+35, -4.1387686369226634e+35, -3.753695009979229e+35, -3.3412218464649816e+35, -3.0061801301109327e+35, -2.7788246745664607e+35, -2.596120612983788e+35, -2.485766133914773e+35, -2.489969764793366e+35, -2.5008328107555816e+35, -2.4960856831028984e+35, -2.4758279809982415e+35, -2.3541916497960662e+35, -2.273211299200904e+35, -2.1837621271882527e+35, -2.094211384422307e+35, -2.104520007068832e+35, -2.0196318043962884e+35, -1.9319019772396167e+35, -1.887200025092215e+35, -1.9275897939260777e+35, -2.0266907539544433e+35, -2.150675356264892e+35, -2.2442595108148387e+35, -2.247855430632049e+35, -2.282279925834427e+35, -2.4135632979790724e+35, -2.513306847387054e+35, -2.4434474147245027e+35, -2.4561254781391895e+35, -2.549989487071126e+35, -2.1216290347929667e+35]
tiempo_s_1emin4 = [0.0, 3.335640952001624e-05, 6.6712819040109e-05, 0.00010006922856020176, 0.00013342563808020223, 0.00016678204760002394, 0.00020013845711984565, 0.00023349486663966736, 0.00026685127615912, 0.0003002076856783996, 0.00033356409519767923, 0.00036692050471695884, 0.00040027691423623845, 0.00043363332375551805, 0.00046698973327479766, 0.0005003461427944694, 0.0005337025523148332, 0.000567058961835197, 0.0006004153713555609, 0.0006337717808759247, 0.0006671281903962885, 0.0007004845999166523, 0.0007338410094370161, 0.0007671974189573799, 0.0008005538284777437, 0.0008339102379981075, 0.0008672666475184714, 0.0009006230570388352, 0.000933979466559199, 0.0009673358760795628, 0.0010006922856014952, 0.0010340486951240274, 0.0010674051046465596, 0.0011007615141690919, 0.001134117923691624, 0.0011674743332141563, 0.0012008307427366885, 0.0012341871522592207, 0.001267543561781753, 0.0013008999713042852, 0.0013342563808268174, 0.0013676127903493496, 0.0014009691998718818, 0.001434325609394414, 0.0014676820189169462, 0.0015010384284394785, 0.0015343948379620107, 0.0015677512474845429, 0.001601107657007075, 0.0016344640665296073, 0.0016678204760521395, 0.0017011768855746717, 0.001734533295097204, 0.0017678897046197362, 0.0018012461141422684, 0.0018346025236648006, 0.0018679589331873328, 0.001901315342709865, 0.0019346717522323973, 0.0019680281617529918, 0.002001384571271187, 0.0020347409807893826, 0.002068097390307578, 0.0021014537998257734, 0.002134810209343969, 0.0021681666188621642, 0.0022015230283803596, 0.002234879437898555, 0.0022682358474167504, 0.002301592256934946, 0.0023349486664531413, 0.0023683050759713367, 0.002401661485489532, 0.0024350178950077275, 0.002468374304525923, 0.0025017307140441183, 0.0025350871235623137, 0.002568443533080509, 0.0026017999425987045, 0.0026351563521169, 0.0026685127616350953, 0.0027018691711532907, 0.002735225580671486, 0.0027685819901896815, 0.002801938399707877, 0.0028352948092260724, 0.0028686512187442678, 0.002902007628262463, 0.0029353640377806586, 0.002968720447298854, 0.0030020768568170494, 0.003035433266335245, 0.00306878967585344, 0.0031021460853716356, 0.003135502494889831, 0.0031688589044080264, 0.003202215313926222, 0.0032355717234444172, 0.0032689281329626126, 0.003302284542480808, 0.0033356409519990035, 0.003368997361517199, 0.0034023537710353943, 0.0034357101805535897, 0.003469066590071785, 0.0035024229995899805, 0.003535779409108176, 0.0035691358186263713, 0.0036024922281445667, 0.003635848637662762, 0.0036692050471809575, 0.003702561456699153, 0.0037359178662173483, 0.0037692742757355437, 0.003802630685253739, 0.0038359870947719346, 0.00386934350429013, 0.0039026999138083254, 0.0039360563233187705, 0.003969412732828292, 0.004002769142337814, 0.012008307424061053]

net_absorption_emission_protons_inv_s_inv_cm3_1emin3 = [0.0, -1.8404626767609604e+35, -1.573148303717554e+35, -1.395278790302969e+35, -1.2845455014078956e+35, -1.2127008949115338e+35, -1.1639979966284877e+35, -1.1297228000765213e+35, -1.104840750384105e+35, -1.0863065016182124e+35, -1.0722068782677536e+35, -1.0613237731174239e+35, -1.0529602460163545e+35, -1.047100081537209e+35, -1.0454816853275151e+35, -1.0559338575905209e+35, -1.1088783821136963e+35, -1.3187828215587193e+35, -2.0873193302424707e+35, -4.008776667018204e+35, -4.4542226100162514e+35, -3.731350306313268e+35, -3.2425889895916702e+35, -3.0055118476067793e+35, -3.694873614285822e+35, -3.888636749429743e+35, -3.4548695862876525e+35, -3.355907345975911e+35, -3.115089867665968e+35, -3.026892215771454e+35, -3.0159526815345413e+35, -3.1810971240711512e+35, -3.2961467959322193e+35, -3.195498470114701e+35, -3.0185596141177263e+35, -3.1708361739571647e+35, -3.1827599185417333e+35, -3.1137713675468012e+35, -3.2791763160356153e+35, -3.452467639332379e+35, -3.436627494248502e+35, -3.6982297355578545e+35, -3.942917377825368e+35, -3.764227193582054e+35, -4.004349551571121e+35, -3.800192585769173e+35, -3.6388003406053866e+35, -3.786936358242693e+35, -4.1173926933875004e+35, -3.795023548462734e+35, -3.7613967097107786e+35, -3.638767237568914e+35, -3.673390534886498e+35, -3.42426750722648e+35, -3.284790619773075e+35, -3.34830577001635e+35, -3.2285542826549373e+35, -3.366530889391445e+35, -3.3932367198780035e+35, -3.5253748137832726e+35, -3.8845544323243215e+35, -3.8132993935040084e+35, -3.8974209393443835e+35, -3.902377971436659e+35, -3.6662675547196595e+35, -3.7069371194627786e+35, -4.057277030347027e+35, -4.207799865039489e+35, -3.839543055933197e+35, -3.937893427644418e+35, -4.177567092819591e+35, -3.706365958243711e+35, -3.6021306658092325e+35, -3.45363923237623e+35, -3.6026918512404016e+35, -3.327142419337937e+35, -3.282224648326411e+35, -3.1410085056372044e+35, -3.170756923472397e+35, -3.385433557219465e+35, -3.548402585701222e+35, -3.619583893238395e+35, -4.102682247354574e+35, -3.856329678343986e+35, -3.680925099685603e+35, -3.755185503137111e+35, -3.954117870323265e+35, -4.2269518592889874e+35, -4.159380405270855e+35, -4.198471977060157e+35, -3.966650610156182e+35, -3.794450367636911e+35, -3.744902945711983e+35, -3.6342193068024114e+35, -3.575538012156404e+35, -3.537741622028615e+35, -3.1959860403873113e+35, -2.9309373309606822e+35, -2.7898767834667057e+35, -2.9846052438748692e+35, -2.916546354344303e+35, -2.8603784016366313e+35, -3.0132194056380822e+35, -3.416362933330216e+35, -3.088572455283255e+35, -2.998517909088517e+35, -3.162561178292003e+35, -3.6834153251443506e+35, -4.637354085927599e+35, -4.613503383417037e+35, -3.932234093631498e+35, -3.5801913692206014e+35, -3.2757804015973343e+35, -3.1869380092141224e+35, -3.1880345587337155e+35, -3.1811957361915056e+35, -3.116452521554992e+35, -3.3293182664148395e+35, -3.22459116078946e+35, -3.021565251710147e+35, -2.7040456100655267e+35, -2.5568476295451416e+35, -2.609528931320764e+35, -2.7683598563084485e+35, -2.8868188365778934e+35, -3.2457656423611005e+35, -3.2422974861169913e+35, -3.022511241189765e+35, -2.7983509849466254e+35, -2.5381226580316283e+35, -2.4029068942637717e+35, -2.432862565086631e+35, -2.4715470900375213e+35, -2.52845791880819e+35, -2.6902363133401466e+35, -2.873159151374292e+35, -3.0405474921054417e+35, -3.0980864753658115e+35, -2.9532893399201142e+35, -3.0339248815736097e+35, -3.341801606066233e+35, -3.622551639505925e+35, -3.6389550010019275e+35, -3.57947804045083e+35, -3.509010991727378e+35, -3.200003878343165e+35, -2.945072285260923e+35, -2.788947878139645e+35, -2.789424896154575e+35, -3.189690892024241e+35, -3.440978361128477e+35, -3.49928691360985e+35, -3.4494164983126946e+35, -3.3069013560052e+35, -3.2151592677890853e+35, -3.0148636579820587e+35, -3.0078137535884692e+35, -2.8590121078613233e+35, -2.7657376645886927e+35, -2.5881281884431313e+35, -2.4424885092598942e+35, -2.3339742238152446e+35, -2.3127104174367166e+35, -2.1586419332458207e+35, -2.0815231206299654e+35, -2.021394080334154e+35, -1.9678751718029747e+35, -1.9330795865400045e+35, -1.922075179122406e+35, -1.9621314816412292e+35, -1.9562367457561533e+35, -1.9962645350391443e+35, -1.9959335828775324e+35, -1.923257639343411e+35, -1.8928696524672273e+35, -1.891965491098639e+35, -1.8840281871001446e+35, -1.8935069169971783e+35, -1.90502715219576e+35, -1.825747781956859e+35, -1.7781071681263832e+35, -1.7876328070372192e+35, -1.8808661368585555e+35, -1.946507155812024e+35, -1.9422832575073816e+35, -2.066424338617686e+35, -2.3467440857528108e+35, -2.869209664151824e+35, -3.550478030916702e+35, -3.651329723613192e+35, -3.1365361786161625e+35, -2.685849383713086e+35, -2.404822939128583e+35, -2.45516110315174e+35, -2.369583135948295e+35, -2.1207090918511577e+35, -1.9170761361473737e+35, -1.8450578296238675e+35, -1.869896341009608e+35, -1.8997386721049325e+35, -2.009433888541402e+35, -2.1041379349758688e+35, -2.1568884526371308e+35, -2.2544453685444747e+35, -2.1618739778749233e+35, -1.967167851501166e+35, -1.8907889644570234e+35, -1.752005228983218e+35, -1.6585758052717372e+35, -1.60738502840185e+35, -1.7843174607074574e+35, -1.9175806295880497e+35, -1.8775315929911836e+35, -1.8477666034686526e+35, -1.726260977059345e+35, -1.6729540689633724e+35, -1.7006751745685408e+35, -1.785322312281298e+35, -1.7949849603897525e+35, -1.7518517068836924e+35, -1.6602145211063118e+35, -1.5624034110224411e+35, -1.5227831995222563e+35, -1.5106888291991716e+35, -1.506914839471705e+35, -1.4711301685186628e+35, -1.463624585788335e+35, -1.4912629041321576e+35, -1.4756781798073475e+35, -1.4333059438495041e+35, -1.41271817819638e+35, -1.3941957197950522e+35, -1.3775968349059104e+35, -1.4170999066127373e+35, -1.4678559978045235e+35, -1.5068434264595026e+35, -1.4971990805061077e+35, -1.4614540096590126e+35, -1.421575364090296e+35, -1.4322697021058721e+35, -1.4728747226599832e+35, -1.4643574082320339e+35, -1.43360818149053e+35, -1.3888310396366374e+35, -1.3489935523316384e+35, -1.3077644665153386e+35, -1.2910940151797742e+35, -1.3016449818472398e+35, -1.3303781785324166e+35, -1.3626607237657268e+35, -1.3841966671058744e+35, -1.3473182656671116e+35, -1.303083722140112e+35, -1.2644563890794224e+35, -1.2364420995706316e+35, -1.2365353878242286e+35, -1.2680696163032112e+35, -1.315815457901883e+35, -1.3385376333203362e+35, -1.3495952582828468e+35, -1.3462379981459527e+35, -1.3912186409704381e+35, -1.4097664884761047e+35, -1.3909244334792216e+35, -1.398321338627782e+35, -1.4347286280606476e+35, -1.5209357660468219e+35, -1.6149653944685238e+35, -1.6435982132914342e+35, -1.575633557284176e+35, -1.532471105433752e+35, -1.5072419196022761e+35, -1.4698216847908396e+35, -1.43802474038915e+35, -1.4075370429103709e+35, -1.3968791090585576e+35, -1.3973552554889863e+35, -1.44711743636794e+35, -1.4882418384136923e+35, -1.5247738937721054e+35, -1.550390376769209e+35, -1.6180566527047802e+35, -1.6824947579019406e+35, -1.737869530869592e+35, -1.818322756481834e+35, -1.890579039500268e+35, -1.9073802988359562e+35, -2.1124444631154327e+35, -2.3872262802853207e+35, -2.5007869293302608e+35, -3.2668971843889157e+35, -5.396814846051544e+35, -8.70491575861157e+35, -8.228673605310624e+35, -6.4919617949009375e+35, -5.429507357320454e+35, -5.256008380960489e+35, -4.6983173538883135e+35, -4.2239912638859476e+35, -4.37883863540333e+35, -5.554015075870751e+35, -6.520833483108184e+35, -6.214015488834848e+35, -5.260109021521767e+35, -4.8477885003284686e+35, -4.7642676089583415e+35, -4.256193310120508e+35, -3.7571831483496055e+35, -3.408233664539338e+35, -3.198881755061991e+35, -3.204933261005075e+35, -2.9529346042481457e+35, -2.5908593981735413e+35, -2.378474594948561e+35, -2.262182201839117e+35, -2.163292793255543e+35, -2.0944079822144592e+35, -1.9782040632140416e+35, -1.883983376662157e+35, -1.9675143114902705e+35, -1.9664257828650935e+35, -2.0047402988501854e+35, -1.994392107729875e+35, -2.0035981457623844e+35, -1.9254071226351263e+35, -1.9187826299944525e+35, -1.912569628167592e+35, -1.836464851523036e+35, -1.7499291753422505e+35, -1.7362886643300524e+35, -1.7113422123777437e+35, -1.6414388540627256e+35, -1.5872070480239139e+35, -1.601001503689829e+35, -1.584101805991642e+35, -1.5416470491393914e+35, -1.468675540465933e+35, -1.4293748761211862e+35, -1.426089191482744e+35, -1.4038645682384687e+35, -1.3749181766654582e+35, -1.3180381795010824e+35, -1.287499477380645e+35, -1.2542035596686233e+35, -1.2350768075313436e+35, -1.2322115273712027e+35, -1.2316602048009049e+35, -1.2264496389078812e+35, -1.232575105577548e+35, -1.2296372678649495e+35, -1.2360471352557114e+35, -1.2525351769040317e+35, -1.2724461548862255e+35, -1.2644903365194686e+35, -1.224414213271027e+35, -1.2012311372836724e+35, -1.1835586079211465e+35, -1.1815874218423621e+35, -1.1646327244743613e+35, -1.1536578888023722e+35, -1.1430789330054086e+35, -1.1465680292946601e+35, -1.1429728922675683e+35, -1.135010971246055e+35, -1.123568479691719e+35, -1.1342189541268534e+35, -1.1414019593000619e+35, -1.1343746955940823e+35, -1.1499556739228822e+35, -1.1472216622932155e+35, -1.13559810227084e+35, -1.1400468216740933e+35, -1.1549489214783893e+35, -1.1503823080550585e+35, -1.1514903430172438e+35, -1.147247831389503e+35]
tiempo_s_1emin3 = [0.0, 3.2022153139306275e-05, 6.404430627937595e-05, 9.606645941944563e-05, 0.00012808861255902626, 0.00016011076569649385, 0.00019213291883396144, 0.00022415507197142903, 0.0002561772251088966, 0.0002881993782463642, 0.0003202215313838318, 0.00035224368452129937, 0.00038426583765876696, 0.00041628799079623455, 0.00044831014393370214, 0.00048033229707116973, 0.0005123544502086373, 0.0005443766033461049, 0.0005763987564835725, 0.0006084209096210401, 0.0006404430627585077, 0.0006724652158959753, 0.0007044873690334429, 0.0007365095221709105, 0.000768531675308378, 0.0008005538284458456, 0.0008325759815833132, 0.0008645981347207808, 0.0008966202878582484, 0.000928642440995716, 0.0009606645941331836, 0.0009926867472706512, 0.0010247089004081188, 0.0010567310535455864, 0.001088753206683054, 0.0011207753598205215, 0.0011527975129579891, 0.0011848196660954567, 0.0012168418192329243, 0.001248863972370392, 0.0012808861255078595, 0.001312908278645327, 0.0013449304317827947, 0.0013769525849202623, 0.0014089747380577299, 0.0014409968911951974, 0.001473019044332665, 0.0015050411974701326, 0.0015370633506076002, 0.0015690855037450678, 0.0016011076568825354, 0.001633129810020003, 0.0016651519631574706, 0.0016971741162949382, 0.0017291962694324058, 0.0017612184225698733, 0.001793240575707341, 0.0018252627288448085, 0.0018572848819822761, 0.0018893070351197437, 0.0019213291882572113, 0.0019533513413949734, 0.0019853734945740743, 0.0020173956477531753, 0.0020494178009322762, 0.002081439954111377, 0.002113462107290478, 0.002145484260469579, 0.00217750641364868, 0.002209528566827781, 0.002241550720006882, 0.002273572873185983, 0.002305595026365084, 0.002337617179544185, 0.0023696393327232858, 0.0024016614859023867, 0.0024336836390814877, 0.0024657057922605886, 0.0024977279454396896, 0.0025297500986187905, 0.0025617722517978915, 0.0025937944049769924, 0.0026258165581560934, 0.0026578387113351943, 0.0026898608645142953, 0.0027218830176933963, 0.002753905170872497, 0.002785927324051598, 0.002817949477230699, 0.0028499716304098, 0.002881993783588901, 0.002914015936768002, 0.002946038089947103, 0.002978060243126204, 0.003010082396305305, 0.003042104549484406, 0.0030741267026635067, 0.0031061488558426077, 0.0031381710090217087, 0.0031701931622008096, 0.0032022153153799106, 0.0032342374685590115, 0.0032662596217381125, 0.0032982817749172134, 0.0033303039280963144, 0.0033623260812754153, 0.0033943482344545163, 0.0034263703876336172, 0.003458392540812718, 0.003490414693991819, 0.00352243684717092, 0.003554459000350021, 0.003586481153529122, 0.003618503306708223, 0.003650525459887324, 0.003682547613066425, 0.003714569766245526, 0.0037465919194246268, 0.0037786140726037277, 0.0038106362257828287, 0.0038426583789619296, 0.0038746805321410306, 0.003906702685318954, 0.003938724838414788, 0.0039707469915106226, 0.004002769144606457, 0.004034791297702291, 0.004066813450798125, 0.0040988356038939595, 0.004130857756989794, 0.004162879910085628, 0.004194902063181462, 0.004226924216277296, 0.004258946369373131, 0.004290968522468965, 0.004322990675564799, 0.004355012828660633, 0.0043870349817564675, 0.004419057134852302, 0.004451079287948136, 0.00448310144104397, 0.004515123594139804, 0.004547145747235639, 0.004579167900331473, 0.004611190053427307, 0.004643212206523141, 0.0046752343596189756, 0.00470725651271481, 0.004739278665810644, 0.004771300818906478, 0.0048033229720023125, 0.004835345125098147, 0.004867367278193981, 0.004899389431289815, 0.004931411584385649, 0.004963433737481484, 0.004995455890577318, 0.005027478043673152, 0.005059500196768986, 0.0050915223498648205, 0.005123544502960655, 0.005155566656056489, 0.005187588809152323, 0.005219610962248157, 0.005251633115343992, 0.005283655268439826, 0.00531567742153566, 0.005347699574631494, 0.0053797217277273286, 0.005411743880823163, 0.005443766033918997, 0.005475788187014831, 0.0055078103401106655, 0.0055398324932065, 0.005571854646302334, 0.005603876799398168, 0.005635898952494002, 0.005667921105589837, 0.005699943258685671, 0.005731965411781505, 0.005763987564877339, 0.0057960097179731735, 0.005828031871069008, 0.005860054024164842, 0.005892076177260676, 0.00592409833035651, 0.005956120483452345, 0.005988142636548179, 0.006020164789644013, 0.006052186942739847, 0.0060842090958356815, 0.006116231248931516, 0.00614825340202735, 0.006180275555123184, 0.0062122977082190185, 0.006244319861314853, 0.006276342014410687, 0.006308364167506521, 0.006340386320602355, 0.00637240847369819, 0.006404430626794024, 0.006436452779889858, 0.006468474932985692, 0.0065004970860815265, 0.006532519239177361, 0.006564541392273195, 0.006596563545369029, 0.006628585698464863, 0.006660607851560698, 0.006692630004656532, 0.006724652157752366, 0.0067566743108482, 0.0067886964639440345, 0.006820718617039869, 0.006852740770135703, 0.006884762923231537, 0.0069167850763273715, 0.006948807229423206, 0.00698082938251904, 0.007012851535614874, 0.007044873688710708, 0.007076895841806543, 0.007108917994902377, 0.007140940147998211, 0.007172962301094045, 0.0072049844541898795, 0.007237006607285714, 0.007269028760381548, 0.007301050913477382, 0.007333073066573216, 0.007365095219669051, 0.007397117372764885, 0.007429139525860719, 0.007461161678956553, 0.0074931838320523875, 0.007525205985148222, 0.007557228138244056, 0.00758925029133989, 0.0076212724444357244, 0.007653294597531559, 0.007685316750627393, 0.007717338903723227, 0.007749361056819061, 0.007781383209914896, 0.007813405363006022, 0.007845427515935323, 0.007877449668864623, 0.007909471821793924, 0.007941493974723225, 0.007973516127652526, 0.008005538280581826, 0.008037560433511127, 0.008069582586440428, 0.008101604739369729, 0.00813362689229903, 0.00816564904522833, 0.008197671198157631, 0.008229693351086932, 0.008261715504016233, 0.008293737656945533, 0.008325759809874834, 0.008357781962804135, 0.008389804115733436, 0.008421826268662736, 0.008453848421592037, 0.008485870574521338, 0.008517892727450639, 0.00854991488037994, 0.00858193703330924, 0.008613959186238541, 0.008645981339167842, 0.008678003492097143, 0.008710025645026443, 0.008742047797955744, 0.008774069950885045, 0.008806092103814346, 0.008838114256743647, 0.008870136409672947, 0.008902158562602248, 0.008934180715531549, 0.00896620286846085, 0.00899822502139015, 0.009030247174319451, 0.009062269327248752, 0.009094291480178053, 0.009126313633107353, 0.009158335786036654, 0.009190357938965955, 0.009222380091895256, 0.009254402244824557, 0.009286424397753857, 0.009318446550683158, 0.009350468703612459, 0.00938249085654176, 0.00941451300947106, 0.009446535162400361, 0.009478557315329662, 0.009510579468258963, 0.009542601621188264, 0.009574623774117564, 0.009606645927046865, 0.009638668079976166, 0.009670690232905467, 0.009702712385834767, 0.009734734538764068, 0.009766756691693369, 0.00979877884462267, 0.00983080099755197, 0.009862823150481271, 0.009894845303410572, 0.009926867456339873, 0.009958889609269174, 0.009990911762198474, 0.010022933915127775, 0.010054956068057076, 0.010086978220986377, 0.010119000373915677, 0.010151022526844978, 0.010183044679774279, 0.01021506683270358, 0.01024708898563288, 0.010279111138562181, 0.010311133291491482, 0.010343155444420783, 0.010375177597350084, 0.010407199750279384, 0.010439221903208685, 0.010471244056137986, 0.010503266209067287, 0.010535288361996588, 0.010567310514925888, 0.010599332667855189, 0.01063135482078449, 0.01066337697371379, 0.010695399126643091, 0.010727421279572392, 0.010759443432501693, 0.010791465585430994, 0.010823487738360294, 0.010855509891289595, 0.010887532044218896, 0.010919554197148197, 0.010951576350077498, 0.010983598503006798, 0.011015620655936099, 0.0110476428088654, 0.0110796649617947, 0.011111687114724001, 0.011143709267653302, 0.011175731420582603, 0.011207753573511904, 0.011239775726441205, 0.011271797879370505, 0.011303820032299806, 0.011335842185229107, 0.011367864338158408, 0.011399886491087708, 0.01143190864401701, 0.01146393079694631, 0.01149595294987561, 0.011527975102804911, 0.011559997255734212, 0.011592019408663513, 0.011624041561592814, 0.011656063714522115, 0.011688085867451415, 0.011720108020380716, 0.011752130173310017, 0.011784152326239318, 0.011816174479168618, 0.01184819663209792, 0.01188021878502722, 0.01191224093795652, 0.011944263090885822, 0.011976285243815122, 0.012008307396744423]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(tiempo_s_1emin3, net_absorption_emission_protons_inv_s_inv_cm3_1emin3, label=r"$10^{-3}$")
ax.plot(tiempo_s_1emin4, net_absorption_emission_protons_inv_s_inv_cm3_1emin4, label=r"$10^{-4}$")
ax.plot(tiempo_s_1emin5, net_absorption_emission_protons_inv_s_inv_cm3_1emin5, label=r"$10^{-5}$")
ax.axhline(y=0, color='black', linestyle='--', linewidth=1)
ax.set_xlim(0.0, 0.005)
ax.set_xlabel(r'$t\,[\mathrm{s}]$')
ax.set_ylabel(r'$\dot{Y}_e \, n_b = \dot{n}_p = \dot{n}_{\bar{\nu}_e} - \dot{n}_{\nu_e} \, [\mathrm{cm}^{-3}\mathrm{s}^{-1}]$')
ax.grid(True, which='both', axis='both', linestyle='--', alpha=0.6)
ax.set_title('Ye time derivative.\n But it is not integrated over time in the \nsimulation since we use a frozen fluid.\n')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=28)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.savefig(f"plots/{finaldir}_n_dot.pdf", bbox_inches='tight')
plt.show()
plt.close(fig)